# Testing BridGE functions to work with AoU data

This is organized by the job that is run in the main bridge.py file.

Convert plink1 file formate to plink2

```bash
# assuming you are in the base directory of this repo

# get the same version of plink2
wget https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_avx2_20260504.zip
unzip plink2_linux_avx2_20260504.zip
rm plink2_linux_avx2_20260504.zip

# set up raw directory with refdata and gwas data

# convert plink1 file into plink2 file
./plink2 --bfile data/raw/gwas_subset --make-pgen --out data/intermediate/gwas

# run some basic filtering and export the raw file to load similarily as original BridGE implementation
./plink2 -pfile data/intermediate/gwas --snps-only just-acgt --exclude-palindromic-snps --autosome --geno 0.01 --maf 0.05 --make-pgen --out data/intermediate/gwas_subset
./plink2 -pfile data/intermediate/gwas_subset --export A --out data/intermediate/gwas_subset

# other steps used in AoU analysis
# --set-all-var-ids @:#:\$r:\$a
# --hwe 0.000001 0.001 midp keep-fewhet
# --indep-pairwise 50 5 0.1
# filtered out snps not near genes 
# calculate per chrom LD
```

## Run Old DataProcess in new python environment to create updated pickle saved data

```bash
conda activate bridge-aou
cd BridGE-Python-AoU
source setup.sh

# python bridge-old.py --projectDir=testing --job=DataProcess --plinkFile=gwas_data_final --geneAnnotation=glist-hg38 --genesets=c2.cp.v7.1

time python bridge.py --projectDir=testing --job=ComputeInteraction --model=combined --nWorker=30 --njobs=4 --R=5
# 34m 18s

# python bridge.py --projectDir=testing --job=ComputeStats --model=combined --nWorker=10 --snpPerms=100 --minPath=10 --R=5

# python bridge.py --projectDir=testing --job=ComputeFDR --model=combined --pvalueCutoff=0.05 --minPath=10 --samplePerms=5

# python bridge.py --projectDir=testing --job=Summarize --model=combined --fdrcut=0.25 --snpPathFile=snp_pathway_min10_max300.pkl

```

In [ ]:
import sys
from os import path
# from datatools import plink2pkl as p2p
# from datatools import bindataa as ba
# from datatools import msigdb2pkl as msig2p
# from datatools import mapsnp2gene as snp2gene
# from datatools import snppathway as snpp
# from datatools import bpmind as bpm
# from corefuns import matrix_operations_par as ci
# from corefuns import genstats_perm as gs
# from corefuns import fdrsampleperm as fdr
# from corefuns import collectresults as cl
# import datetime
# import multiprocessing as mp

# bridge.py all possible input args
job = ''
plinkfile = 'gwas_data_final'
project_dir = 'testing'
genesets = 'c2.cp.v7.1' 
gene_annotation = 'glist-hg38'
mappingDistance = 50000
minPath = 10
maxPath = 300
alpha1 = 0.05
alpha2 = 0.05
n_workers = 30
sample_perms = 10
binaryNetwork = False
snpPerms = 10000
i = -1
r = 0
pval_cutoff = 0.05
fdrcut = 0.25
densitycutoff = None
ssmfile = None
model = 'combined'
snppathwayfile = 'snp_pathway_min10_max300.pkl'

## Testing Sparsity and Mat Mul

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix
from sparse import GCXS as sparse_coo
import time

def benchmark_sparsity(shape, sparsity_percent, iters=100):
    """Compare sparse vs dense for different sparsity levels"""
    n_nonzero = int(shape[0] * shape[1] * (100 - sparsity_percent) / 100)
    
    # Create dense array
    dense = np.zeros(shape)
    indices = np.random.choice(shape[0] * shape[1], n_nonzero, replace=False)
    dense.flat[indices] = np.random.rand(n_nonzero)

    # Benchmark matrix-vector product
    csr_arr = csr_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = dense @ dense.T
    dense_time = time.perf_counter() - t0
    dense_mem = dense.data.nbytes
    print(f"dense={dense_mem/1e6:.1f}MB")

    csc_arr = csc_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = csr_arr @ csr_arr.T
    csr_time = time.perf_counter() - t0
    speedup_csr = dense_time / csr_time
    csr_mem = csr_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_csr:.2f}x, csr={csr_mem/1e6:.1f}MB")
    
    coo_arr = coo_matrix(dense)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = csc_arr @ csc_arr.T
    csc_time = time.perf_counter() - t0
    speedup_csc = dense_time / csc_time
    csc_mem = csc_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_csc:.2f}x, csc={csc_mem/1e6:.1f}MB")
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = coo_arr @ coo_arr.T
    coo_time = time.perf_counter() - t0
    speedup_coo = dense_time / coo_time
    coo_mem = coo_arr.data.nbytes
    print(f"Sparsity {sparsity_percent}%: speedup={speedup_coo:.2f}x, coo={coo_mem/1e6:.1f}MB")
    
    # sparse_coo_arr = sparse_coo(dense)
    # t0 = time.perf_counter()
    # for _ in range(iters):
    #     _ = sparse_coo_arr @ sparse_coo_arr.T
    # sparse_coo_time = time.perf_counter() - t0
    # speedup_sparse_coo = dense_time / sparse_coo_time
    # sparse_coo_mem = sparse_coo_arr.data.nbytes
    # print(f"Sparsity {sparsity_percent}%: speedup={speedup_sparse_coo:.2f}x, sparse_coo={sparse_coo_mem/1e6:.1f}MB")
    
    print()

# Test at different sparsity levels
shape = (10000, 100)
for sparsity in [90, 95, 99]:  # 50, 70
    benchmark_sparsity(shape, sparsity, iters=10)


In [ ]:
import numpy as np
import time

def benchmark_matmul(size=2048, iterations=10):
    """
    Benchmarks NumPy matrix multiplication for float16, float32, and float64.
    
    Parameters:
    size (int): The dimensions of the square matrices (size x size).
    iterations (int): How many times to run the multiplication for an accurate average.
    """
    
    # Define the data types and their display names
    dtypes = [np.float32, np.float64]  # np.float16 is very slow
    names = ["Single (float32)", "Double (float64)"]
    
    print(f"Benchmarking NumPy MatMul ({size} x {size}) - Average of {iterations} runs\n")
    print(f"{'Precision':<20} | {'Avg Time (Seconds)':<20}")
    print("-" * 45)
    
    for dtype, name in zip(dtypes, names):
        # Generate random matrices and cast them to the specific precision
        A = np.random.rand(size, size).astype(dtype)
        B = np.random.rand(size, size).astype(dtype)
        
        # Warm-up run to initialize memory and avoid first-run overhead
        _ = A @ B
        
        # Start benchmarking
        start_time = time.perf_counter()
        for _ in range(iterations):
            _ = A @ B
        end_time = time.perf_counter()
        
        # Calculate and print the average time
        avg_time = (end_time - start_time) / iterations
        print(f"{name:<20} | {avg_time:.5f} s")

if __name__ == "__main__":
    # You can adjust the matrix size and iterations here
    benchmark_matmul(size=1000, iterations=100)
    # float16 was super slow
    # overall, float32 is faster than float64

## DataProcess

In [ ]:
job = 'DataProcess'



if job == 'DataProcess':
    print('data processing...')
    sys.stdout.flush()

    # convert plinkfile to pickle
    if plinkfile == '':
        sys.exit('plinkFile not provided')
    rawfile = f"{project_dir}/intermediate/{plinkfile}.raw"
    pvarfile = f"{project_dir}/intermediate/{plinkfile}.pvar"
    psamfile = f"{project_dir}/intermediate/{plinkfile}.psam"
    if not path.exists(rawfile) or not path.exists(pvarfile) or not path.exists(psamfile):
        sys.exit(f'plinkFiles do not exist:\n\t{rawfile},\n\t{pvarfile},\n\t{psamfile}')
    finalfile = f"{project_dir}/intermediate/{plinkfile}.pkl"
    # p2p.plink2pkl(pgenfile, pvarfile, psamfile, finalfile)

    # converting snp data assuming different disease models
    # ba.bindataa(project_dir, finalfile, 'r')
    # ba.bindataa(project_dir, finalfile, 'd')

    # prepare gene set information
    symbolsfile = f"{project_dir}/raw/{genesets}.symbols.gmt"
    entrezfile = f"{project_dir}/raw/{genesets}.entrez.gmt"
    if not path.exists(symbolsfile) or not path.exists(entrezfile):
        sys.exit(f'genesets do not exist: {symbolsfile}, {entrezfile}')
    # msig2p.msigdb2pkl(symbolsfile, entrezfile)

    # build relationship between snps and genes
    gene_annotation_file = f"{project_dir}/raw/{gene_annotation}"
    if not path.exists(gene_annotation_file):
        sys.exit('gene annotation file not found')
    sgmfile = f"{project_dir}/intermediate/snpgenemapping_{int(mappingDistance/1000)}kb.pkl"
    # snp2gene.mapsnp2gene(pvarfile, gene_annotation_file, mappingDistance, 'matrix', sgmfile) # matrix mode - #change

    # extract snp-pathway information
    geneset_pkl = f"{project_dir}/intermediate/{genesets}.pkl"
    # outfile = snpp.snppathway(finalfile, sgmfile, geneset_pkl, minPath, maxPath)
    # bpm.bpmind(outfile)

### datatools/plink2pkl.py - done

In [ ]:
import pickle

import pandas as pd

from datatools import imputesnp as isnp
from classes import SNPdataclass as snpc


def assess_sparseness(df):
    """Sparsity as percentage of missing/null values"""
    
    sparsity = df.isnull().sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (missing/nulls): {sparsity:.2%}")

    # Count zeros as sparse (common in genomics)
    sparsity = (df == 0).sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (zeros): {sparsity:.2%}")

    # Or combine zeros and nulls
    sparsity = ((df == 0) | df.isnull()).sum().sum() / (len(df) * len(df.columns))
    print(f"Sparsity (zeros + missing): {sparsity:.2%}")
    
def plink2pkl(rawFile, pvarFile, psamFile, outputFile):
    """Convert plink .pgen file to pickle file format.
        
    This function extracts all information from the .pgen file and 
    separates the genotype information from the rest. It saves all 
    into a <outputFile>.pkl file.
    
    INPUTS:
    rawFile - plink.raw file
    pvarFile - plink.pvar file that associated with rawFile
    psamFile - plink.psam file that associated with rawFile
    outputFile - name for output pickle file
    
    OUTPUTS:
    <outputFile>.pkl
    The .pkl file uses an SNPdata class with the following fields:
    - rsid: snp names
    - data: genotype data
    - chr: chromosome id
    - loc: physical location
    - pheno: sample's phenotype
    - fid: sample's family id
    - pid: sample id
    - sex: sample sex
    """
    
    # Creating headers for columns reading files into dataframes.
    pvar_header = ['chrom', 'pos', 'var_id', 'ref', 'alt']
    var_df = pd.read_csv(pvarFile, sep=r"\s+", header=0, names=pvar_header, engine='python')

    psam_header = ['fid', 'iid', 'sex', 'pheno']
    sam_df = pd.read_csv(psamFile, sep=r"\s+", header=0, names=psam_header, engine='python')

    geno_df = pd.read_csv(rawFile, sep=r"\s+", header=0, engine='python')

    # need to flip 0 and 2 counts, since plink's --export A counts the ref alleles
    data = 2 - geno_df[geno_df.columns[6:]]
    assess_sparseness(data)

    # remove ref allele from the end of the rsIDs
    prev_cols = data.columns.tolist()
    new_cols = [col.split('_')[0] for col in prev_cols]
    data.columns = new_cols

    # Structuring data to be saved into pickle format.
    SNPdata = snpc.SNPclass(
        data, 
        var_df.var_id, var_df.chrom, var_df.pos,
        sam_df.pheno-1, sam_df.fid, sam_df.iid, sam_df.sex,
        )

    # Save data to pickle file.
    with open(outputFile, 'wb') as file:
        pickle.dump(SNPdata, file, protocol=pickle.HIGHEST_PROTOCOL)

plink2pkl(rawfile, pvarfile, psamfile, finalfile)

### datatools/bindata.py - done

In [ ]:
import pickle


def bindataa(project_dir, dataFile, expr):
    """Binarize 012 format SNP data based on dominant/recessive assumptions.

    INPUTS:
    project_dir: directory of all the project files
    dataFile - name of the data file. This .mat file consists
        a structure array SNPdata with the following fields:
        - rsid:snp names
        - data:genotype data
        - chr: chromosome id
        - loc: physical location
        - pheno: sample's phenotype
        - fid: sample's family id
        - pid: sample id
        - gender
    expr - flag used to designate dominant ('d'/'D') or recessive ('r'/'R')

    OUTPUTS:
    a pickle file SNPdataA(D or R).pkl
    """
    
    # Reading in pickle datafile
    pklin = open(dataFile, "rb")
    SNPdata = pickle.load(pklin)
    pklin.close()

    # Checking expression flag to proceed as dominant or recessive (D or R).
    if expr == 'r' or expr == 'R':
        # If recessive, set 1s to 0s, 2s to 1s, and set appropriate filename.
        filename = f"{project_dir}/intermediate/SNPdataAR.pkl"
        replace_dict = {1: 0, 2: 1}
        SNPdata.data = SNPdata.data.replace(replace_dict)
        
    elif expr == 'd' or expr == 'D':
        # If dominant, set 1s to 1s, 2s to 1s, and set appropriate filename.
        filename = f"{project_dir}/intermediate/SNPdataAD.pkl"
        replace_dict = {2: 1}
        SNPdata.data = SNPdata.data.replace(replace_dict)
        
    else:
        # Default case where expression provided was neither D or R
        print("Provide 'd'/'D' or 'r'/'R' to designate dominant/recessive.")
        return

    # TODO: this redundantly saves SNPdata class, however, it would be easy to simply
    # change the data when either dominant or recessive is needed.
    # Will need to make sure that there aren't other changes to the SNPdata class other than the data
    
    # Saving updated SNPdata in output pickle file.
    final = open(filename, 'wb')
    pickle.dump(SNPdata, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()


In [ ]:
bindataa(project_dir, finalfile, 'r')

In [ ]:
bindataa(project_dir, finalfile, 'd')

### datatools/msigdb2pkl.py - done

In [ ]:
import pickle

import numpy as np
import pandas as pd

from classes import genesetdataclass as gsc


def msigdb2pkl(symbolsFile, entrezFile):
    """Convert MsigDB gene set file (.gmt) to pickle file (Python pkl).
        
    Args:
        symbolsFile: MsigDB gene set file using gene symbols (.symbols.gmt).
        entrezFile: MsigDB gene set file using gene entrez ids (.entrez.gmt).
        
    OUTPUTS:
        <symbolsFile>.pkl - This pickle file uses a geneset class with fields:
            geneset.entrezids - gene {symbol: entrezID} lookup dictionary
            geneset.gpmatrix - gene pathway binary dataframe
    """
    
    # load pathway files
    symbols_df = pd.read_csv(symbolsFile, header=None)
    symbols_df = symbols_df[0].str.split('\t', expand=True, n=2)
    symbols_df.columns = ['pathway_names', "url", "gene_names"]
    symbols_df['gene_names'] = symbols_df['gene_names'].str.split('\t')

    entrez_df = pd.read_csv(entrezFile, header=None)
    entrez_df = entrez_df[0].str.split('\t', expand=True, n=2)
    entrez_df.columns = ['pathway_names', "url", "entrez_ids"]
    entrez_df['entrez_ids'] = entrez_df['entrez_ids'].str.split('\t')

    # make gene by pathway binary matrix
    pathway_list = symbols_df['pathway_names'].tolist()
    gene_list = list(set([gene for sublist in symbols_df['gene_names'].tolist() for gene in sublist]))
    gpm = pd.DataFrame(np.zeros((len(gene_list), len(pathway_list))),
                            index=pd.Series(gene_list, name='genes'),
                            columns=pd.Series(pathway_list, name='pathway'),
                            dtype=bool)
    
    # fill out binary matrix
    for pathway in pathway_list:
        pathway_mask: pd.Series = symbols_df['pathway_names'] == pathway
        genes_in_pathway = symbols_df.loc[pathway_mask, 'gene_names'].tolist()[0]
        gpm.loc[genes_in_pathway, pathway] = True
    
    # Creating dictionary for easy lookup of entrezID by symbol.
    symboldict = {}
    for symbol_genes, entrez_ids in zip(symbols_df['gene_names'].tolist(), entrez_df['entrez_ids'].tolist()):
        for symbol, entrez_id in zip(symbol_genes, entrez_ids):
            symboldict[symbol] = int(entrez_id)

    # 6/25/26 MF - confirmed this gene by pathway matrix is correct and matches the original implementation
    # Converting data to pickle storage file with geneset class.
    geneset = gsc.genesetclass(symboldict, gpm)
    symbols_pkl_file = symbolsFile.replace(".symbols.gmt", ".pkl")
    symbols_pkl_file = symbols_pkl_file.replace("raw/", "intermediate/")
    final = open(symbols_pkl_file, 'wb')
    pickle.dump(geneset, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()


msigdb2pkl(symbolsfile, entrezfile)

### datatools/mapsnp2gene.py - done

In [ ]:
import pickle

import numpy as np
import pandas as pd


def mapsnp2gene(pvarFile, geneAnnotation, mappingDistance, option, outfile):
    """Creates snp to gene matrix in the DataFrame format and saves it to a pickle file.

    Args:
        pvarFile (str): path to Plink variant file in .pvar format.
        geneAnnotation (str): path to gene annotation file.
        mappingDistance (int): snp to gene mapping distance.
        option (str): saving mode for snp-gene map.
        outfile (str): file name for saving the results.
    """

    # Creating SNP dataframe from snp annotation file.
    pvar_header = ['chrom', 'pos', 'var_id', 'ref', 'alt']
    var_df = pd.read_csv(pvarFile, sep=r"\s+", header=0, names=pvar_header, engine='python')  # has header row
    var_df['chrom'] = pd.to_numeric(var_df['chrom'])

    # Creating gene dataframe from gene annotation file.
    gene_header = ['chrom', 'geneloc1', 'geneloc2', 'genes']
    gdf = pd.read_csv(geneAnnotation, sep=r"\s+", names=gene_header, engine='python')  # does not have a header
    gdf = gdf[gdf.chrom.apply(lambda x: x.isnumeric())]
    gdf['chrom'] = pd.to_numeric(gdf['chrom'])
    gdf.sort_values(by='chrom', inplace=True)

    # Expanding gene window by subtracting and adding from start and end loci.
    gdf['geneloc1'] = gdf['geneloc1'] - mappingDistance
    gdf['geneloc2'] = gdf['geneloc2'] + mappingDistance

    # Doing an outer join to get all genes and snp listed by chromosome.
    cdf = gdf.merge(var_df, how='outer', on='chrom')

    # keep only snps that are located between start and end loci adjusted by mappingDistance.
    cdf = cdf[(cdf['pos'] >= cdf['geneloc1']) & (cdf['pos'] <= cdf['geneloc2'])]
    
    # Creating list of unique rsids from filtered results.
    snplist = cdf['var_id'].drop_duplicates()

    # Option chosen to save to snplist.
    if (option == 'snplist'):

        # Saving SNPlist to pickle file.
        final = open(outfile, 'wb')
        pickle.dump(snplist, final, protocol=pickle.HIGHEST_PROTOCOL)
        final.close()

    # Option chosen to save to matrix.
    elif (option == 'matrix'):

        # Getting list of unique and genes from filtered results.
        genelist = cdf['genes'].drop_duplicates()

        # Creating dataframe of appropriate size, and setting labels.
        sgm = pd.DataFrame(np.zeros((len(snplist), len(genelist))),
                                index=snplist, columns=genelist, dtype=bool)

        # Setting snp-gene matrix values to true if snp is within gene window.
        for row in cdf.itertuples():
            sgm.loc[row.var_id, row.genes] = True

        # Saving snp-gene matrix to pickle file.
        final = open(outfile, 'wb')
        pickle.dump(sgm, final, protocol=pickle.HIGHEST_PROTOCOL)
        final.close()

    else:
        # Output option not recognized.
        print("Return option error, valid options are 'snplist', or 'matrix'")


mapsnp2gene(pvarfile, gene_annotation_file, mappingDistance, 'matrix', sgmfile)
# 50s

### datatools/snppathway.py - done

In [ ]:
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_array

from classes import snpsetclass as snps


def snppathway(dataFile, sgmFile, genesets, minPath, maxPath):
    """Creates a snp-to-pathway mapping a snpset class object with following fields:
        - pathways: List of pathway names
        - spmatrix: Matrix of snp-pathway mapping (Numpy 2d array)
        - geneset: Path to geneset file in .pkl format.

    Args:
        dataFile (str): Path to the file with genotype data in the Pickle format.
        sgmFile (str): SNP to gene mapping file in the Pickle format.
        genesets (str): Gene-set file in pickle format.
        minPath (int): Minimum size for a pathway to be in the mapping.
        maxPath (int): Maximum size for a pathway to be in the mapping.

    Returns:
        str: Path to the output pickle file containing the snp-to-pathway mapping.
    """
    
    # find project directory
    p_dir = dataFile.split('/')
    s = '/'
    project_dir = s.join(p_dir[0:-1])

    # Loading pickle files into objects
    pklin = open(dataFile, "rb")
    SNPdata = pickle.load(pklin)
    pklin.close()

    pklin = open(sgmFile, "rb")
    sgm = pickle.load(pklin)  # snps are rows, genes are columns
    pklin.close()

    pklin = open(genesets, "rb")
    geneset = pickle.load(pklin)
    pklin.close()

    # find the snps in SNPdata (plink data) that are also in the snp-gene matrix
    # since the snp-gene matrix was created from the plink data, this is simply a sanity check that runs fast
    tmp_ids = np.intersect1d(SNPdata.rsid, sgm.index)  # TODO: 6/25/26 MF - I need to change this to var_id in SNPClass.py
    ind_ids = sgm.index.isin(tmp_ids)
    tmp_sgm = sgm.loc[ind_ids, :]

    # keep only pathways with total genes less than upper limit and more than lower limit
    ind = (np.sum(geneset.gpmatrix, axis=0) <= maxPath) & (np.sum(geneset.gpmatrix, axis=0) >= minPath)
    tmp_gpm = geneset.gpmatrix.loc[:, ind]

    # keep genes that are in both snp-gene and gene-pathway matrices
    keep_genes = np.intersect1d(tmp_gpm.index, tmp_sgm.columns)
    tmp2_sgm = tmp_sgm.loc[:, keep_genes]
    tmp2_gpm = tmp_gpm.loc[keep_genes, :]

    # make snp-pathway matrix with dot product of sparse arrays (near instant computation)
    sg_sparse = csr_array(tmp2_sgm.to_numpy())
    gp_sparse = csr_array(tmp2_gpm.to_numpy())
    tmp_sgp = sg_sparse.dot(gp_sparse).toarray()

    # after matrix multiplication (dot product) there will be values greater than 1
    # set data type to bool and then back to int
    tmp_sgp = tmp_sgp.astype(bool).astype(int)
    tmp_sgp_df = pd.DataFrame(tmp_sgp,
                                index=pd.Series(tmp2_sgm.index, name='var_id'),
                                columns=pd.Series(tmp2_gpm.columns, name='pathway'))

    # remove pathways with total SNPs more than upper limit and less than lower limit
    ind = (np.sum(tmp_sgp_df, axis=0) <= maxPath) & (np.sum(tmp_sgp_df, axis=0) >= minPath)
    tmp_sgp_df = tmp_sgp_df.loc[:, ind]

    # remove snps (rows) that aren't in a pathway
    ind_rows = (np.sum(tmp_sgp_df, axis=1) == 0)
    remove_rows = tmp_sgp_df.index[ind_rows]
    tmp_sgp_df = tmp_sgp_df.drop(remove_rows, axis=0)

    # remove pathways (columns) that aren't in any snps
    ind_cols = (np.sum(tmp_sgp_df, axis=0) == 0)
    remove_cols = tmp_sgp_df.columns[ind_cols]
    tmp_sgp_df = tmp_sgp_df.drop(remove_cols, axis=1)

    # check again the SNP limit (mostly just for lower bound, but we'll keep in upper bound too)
    ind = (np.sum(tmp_sgp_df, axis=0) <= maxPath) & (np.sum(tmp_sgp_df, axis=0) >= minPath)
    tmp_sgp_df = tmp_sgp_df.loc[:, ind]

    # Preparing data and filename for pickle storage.
    pathways = tmp_sgp_df.sum(axis=0)
    snpset = snps.snpsetclass(pathways, tmp_sgp_df, genesets)
    outfilename = f"{project_dir}/snp_pathway_min{minPath}_max{maxPath}.pkl"

    # Saving data to pickle file.
    final = open(outfilename, 'wb')
    pickle.dump(snpset, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()

    # Returning the name of the output file to be used by other modules.
    return outfilename
    # 6/25/26 MF - confirmed this snp by pathway matrix is correct BUT DOES NOT MATCH THE ORIGINAL IMPLEMENTATION
    # see AoU-run_bridge.md for details.


outfile = snppathway(finalfile, sgmfile, geneset_pkl, minPath, maxPath)

### datatools/bpmind.py - done

In [ ]:
import pickle
from itertools import combinations

import numpy as np
import pandas as pd

from classes import bpmindclass as bpmc


def bpmind(snpPathwayFile):
    """Exctracts SNP indices for BPM/WPM sets. Saves a BPMind.pkl file with a bpmindclass class with fields:
        bpm - DataFrame with all BPM data (pathway names, pathway inices, SNPs in pathaways(redundants removed))
        wpm - DataFrame with all WPM data (pathway names, pathway inices, SNPs in pathaways)

    Args:
        snpPathwayFile (str): SNP-pathway mapping file in pickle format (.pkl), containing a matrix: Result of the snppathway function. 
    """

    # find project directory
    p_dir = snpPathwayFile.split('/')
    s = '/'
    project_dir = s.join(p_dir[0:-1])

    # Reading in pickle datafile
    pklin = open(snpPathwayFile, "rb")
    snpset = pickle.load(pklin)
    pklin.close()

    # Retrieving pathways list from snpset
    pathways = snpset.pathways
    snpmat = snpset.spmatrix

    # Finding all possible combinations of pairs for pathway names and sizes.
    combnames = np.array(list(combinations(pathways.index, 2)))

    # Finding nonzero WPM indices
    WPMind = [ np.nonzero(snpmat[column])[0].tolist() for column in snpmat.columns ]
    wpmdata = {
        'pathway': pathways.index,
        'indsize': pathways.values,
        'ind': WPMind,
        'size': ((pathways.values * pathways.values) - pathways.values),  # TODO: 7/6/26 MF - evaluate if this needs to be divided by 2 (n choose k, k=2)
        }
    wpm = pd.DataFrame(wpmdata)

    # Finding BPM indices
    BPMind1, BPMind2, ind1size, ind2size = [], [], [], []
    for i in range(len(snpmat.columns)):
        p1 = snpmat.iloc[:, i].to_numpy()
        
        for j in range(i + 1, len(snpmat.columns)):
            p2 = snpmat.iloc[:, j].to_numpy()
            
            # snps in pathway 1 but not in pathway 2
            d1 = p1 - p2
            ind1 = np.where(d1 == 1)[0].tolist()
            
            # snps in pathway 2 but not in pathway 1
            d2 = p2 - p1
            ind2 = np.where(d2 == 1)[0].tolist()
            
            ind1size.append(len(ind1))
            ind2size.append(len(ind2))
            
            BPMind1.append(ind1)
            BPMind2.append(ind2)

    # Getting between pathway sizes by multiplying combination available pairs.
    if (len(pathways) > 1):
        size = np.array(ind1size) * np.array(ind2size)
        # Orienting bpm/wpm data and converting to dataframes.
        bpmdata = {
            'path1names': combnames[:, 0], 'ind1size': ind1size, 'ind1': BPMind1,
            'path2names': combnames[:, 1], 'ind2size': ind2size, 'ind2': BPMind2,
            'size': size,
            }
    else:
        bpmdata = {
            'path1names': [], 'ind1size': [],
            'path2names': [], 'ind2size': [],
            'size': [],
            }
    bpm = pd.DataFrame(bpmdata)

    # Reading bpm and wpm models into bpmind class for pickle storage.
    bpmobj = bpmc.bpmindclass(bpm, wpm)

    # Saving bpmind data to pickle file.
    final = open(project_dir+'/BPMind.pkl', 'wb')
    pickle.dump(bpmobj, final, protocol=pickle.HIGHEST_PROTOCOL)
    final.close()

bpmind(outfile)

In [ ]:
snpPathwayFile = outfile

# find project directory
p_dir = snpPathwayFile.split('/')
s = '/'
project_dir = s.join(p_dir[0:-1])

# Reading in pickle datafile
pklin = open(snpPathwayFile, "rb")
snpset = pickle.load(pklin)
pklin.close()

# Retrieving pathways list from snpset
pathways = snpset.pathways

In [ ]:
pathways.values

In [ ]:
((pathways.values * pathways.values) - pathways.values) 

In [ ]:
((pathways.values * pathways.values) - pathways.values) / 2

## ComputeInteraction

In [ ]:
job = 'ComputeInteraction'





if job == 'ComputeInteraction':
    if not (model == 'RR' or model == 'RD' or model == 'DD' or model == 'combined'):
        sys.exit('wrong model')
        
    snpDataAD = f"{project_dir}/intermediate/SNPdataAD.pkl"
    if not path.exists(snpDataAD):
        sys.exit(snpDataAD + ' not found')
        
    snpDataAR = f"{project_dir}/intermediate/SNPdataAR.pkl"
    if not path.exists(snpDataAR):
        sys.exit(snpDataAR + ' not found')
        
    # if r < 0 :
    #     if model == 'combined':
    #         ci.combine(project_dir, alpha1, alpha2, n_workers, i)
    #     else:
    #         ci.run(project_dir, model, alpha1, alpha2, n_workers, i)
    # else:
    #     for i in range(r + 1):
    #         if model == 'combined':
    #             ci.combine(project_dir, alpha1, alpha2, n_workers, i)
    #         else:
    #             ci.run(project_dir, model, alpha1, alpha2, n_workers, i)

### corefuns/matrix_operations_par.py

In [ ]:
import math
import sys
import pickle
from datetime import datetime
from os import path

import numpy as np
import scipy.sparse

from corefuns_new import HygeCache as hc
from corefuns_new import withinclassrand as wrand
from classes import InteractionNetwork


# matrix_operations_par computes the interaction network. The functions to call are run() and combine()
#
# REFACTOR NOTES (see accompanying summary):
#   - Workers no longer write into a giant shared dense (s x s) ctypes array. Each worker
#     returns sparse (row, col, value) triples for its block, which the parent assembles
#     into a scipy.sparse.csr_matrix. Most SNP pairs fail the alpha1/alpha2 filters, so this
#     is a large memory win at any meaningful value of s.
#   - sy is processed in column tiles (sy_chunk_size) inside each worker so peak memory per
#     worker no longer scales with the full s, only with (chunk_rows x sy_chunk_size).
#   - g10/g01/g00/x10/x01/x00 are derived from row/column sums of g11/x11 instead of being
#     computed via separate matmuls, and xp11/xp10/xp01/xp00 = g - x (since pheno_res = 1-pheno
#     is linear). This drops matmuls per chunk from 12 to 2 and removes 8 dense intermediates
#     (Ix, Iy, sx_res, sy_res, tempx_r, temp_r, temp, and the redundant g/x recomputation).
#   - InteractionNetwork now stores scipy.sparse.csr_matrix for risk/protective instead of
#     dense numpy arrays. Downstream consumers (genstats_perm.py, fdrsampleperm.py,
#     collectresults.py) will need to be updated to accept sparse matrices, consistent with
#     the broader sparse-matrix migration already underway in DataProcess.
#
# INPUTS:
#	project_dir: Project directory including all data files
#	model: disease model, can be RR-DD-RD, for combining them, call combine() function instead of run()
#	alpha1: maximum p-value threshold for p11 in combinations
#	alpha2: minimum p-value threshold for p10, p01, p00 in combinations
#	n_workers: Number of CPU cores used for parallel computing
#	R: network number identifier, 0 for real, non-zero for random networks(phenotype labels will be randomly shuffled before computing interactions)
#
# OUTPUTS:
#   ssM_mhygessi_{model}_R{R}.pkl - This pickle file contains an InteractionNetwork class object with following fields:
#       - risk: Risk-associated SNP-SNP interaction scores, scipy.sparse.csr_matrix
#       - protective: Protective SNP-SNP interaction scores, scipy.sparse.csr_matrix
#		- risk_max_id: indicator of which disease model has the maximum risk score for each SNP pair, used in combined model
#		- protective_max_id:  indicator of which disease model has the maximum protective score for each SNP pair, used in combined model
#


def helper_score_from_counts(cache, g11, x11, g10, x10, g01, x01, g00, x00, alpha1, alpha2, risk, pool, n_workers):
    """Reproduces the original p-value/log-score/filter logic, operating on 1D arrays."""

    eps = 1e-10
    p11 = cache.apply_hyge(g11, x11, risk, pool, n_workers) + eps
    p10 = cache.apply_hyge(g10, x10, risk, pool, n_workers) + eps
    p01 = cache.apply_hyge(g01, x01, risk, pool, n_workers) + eps
    p00 = cache.apply_hyge(g00, x00, risk, pool, n_workers) + eps
    q_min = np.minimum(np.minimum(p01, p10), p00)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        log_out = -np.log10(p11 / q_min)
    
    fail = (p11 > alpha1) | (p10 <= alpha2) | (p01 <= alpha2) | (p00 <= alpha2)
    log_out[fail] = 0
    log_out[q_min == 0] = 0
    log_out[~np.isfinite(log_out)] = 0
    log_out[log_out < 0] = 0
    # NOTE: original also had `log_out[p11 == 0] = 0`, but p11 is always >= eps > 0 here
    # (it was dead code in the original too) so it's omitted.
    return log_out

def run(project_dir, model, alpha1, alpha2, n_jobs, n_workers, pool, R):
    """Computes the interaction network for a single model (RR, RD, or DD) and saves it to a pickle file.

    Args:
        project_dir (_type_): _description_
        model (_type_): _description_
        alpha1 (_type_): _description_
        alpha2 (_type_): _description_
        n_jobs (_type_): _description_
        n_workers (_type_): _description_
        R (_type_): _description_
    """
    
    print('computing interaction. R=' + str(R) + ' model = ' + model)
    output_name = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{R}.pkl"
    cluster_file = f"{project_dir}/intermediate/PlinkFile.cluster2"

    # loading and reading SNP data - skipped if caller already loaded these (e.g. looping over R)
    with open(f"{project_dir}/intermediate/SNPdataAD.pkl", "rb") as pkl_d:
        snpdata_d = pickle.load(pkl_d)
    with open(f"{project_dir}/intermediate/SNPdataAR.pkl", "rb") as pkl_r:
        snpdata_r = pickle.load(pkl_r)

    pheno = snpdata_r.pheno
    dataR = snpdata_r.data
    dataD = snpdata_d.data

    symmetric_flag = (model == 'RR' or model == 'DD')

    if model == 'RR':
        datai = dataR
        dataj = dataR
    elif model == 'DD':
        datai = dataD
        dataj = dataD
    else:
        datai = dataR
        dataj = dataD

    population_size = pheno.shape[0]
    ## shuffle phenotypes if R != 0
    if R > 0:
        if not path.exists(cluster_file):
            # single deterministic permutation per R, instead of discarding R-1 throwaway
            # permutations from a shared RNG stream.
            
            # rng = np.random.RandomState(66754 * R)  # TODO: change RandomState to default_rng.
            # permuted_idx = rng.permutation(population_size)
            
            # TODO: remove this once I know that this isn't causing the combined max id issues.
            np.random.seed(66754)
            for i in range(R):
                permuted_idx = np.random.permutation(population_size)
    
            pheno = pheno[permuted_idx]
        else:
            pheno = wrand.withinclassrand(R, cluster_file, f"{project_dir}/intermediate/SNPdataAD.pkl")

    case_size = int(np.count_nonzero(pheno))
    control_size = population_size - case_size
    cache = hc.HygeCache(population_size, case_size, control_size)

    sx_full = np.ascontiguousarray(datai, dtype=np.float32)
    sy_full = np.ascontiguousarray(dataj, dtype=np.float32)
    pheno = np.asarray(pheno, dtype=np.float32).ravel()
    s = sx_full.shape[1]

    ## dividing sx for parallel computing (unchanged balancing logic - earlier chunks have
    ## fewer lower-triangle entries for symmetric models, so chunk boundaries are sqrt-spaced)
    idx = [0]
    if model == 'RR' or model == 'DD':
        share = s * s / n_jobs
        for i in range(n_jobs):
            if i == n_jobs - 1:
                idx.append(s)
            else:
                idx.append(math.floor(math.sqrt(idx[i] * idx[i] + share)))
    else:
        share = math.floor(s / n_jobs)
        for i in range(n_jobs):
            if i == n_jobs - 1:
                idx.append(s)
            else:
                idx.append(idx[i] + share)

    
    results = []
    for i in range(n_jobs):
        t1 = datetime.now()
        
        i1 = idx[i]
        i2 = idx[i + 1]
        sx = np.ascontiguousarray(sx_full[:, i1:i2], dtype=np.float32)
        s = sy_full.shape[1]
        
        print(f"job split {i+1}/{n_jobs}: i1 = {i1}, i2 = {i2}")
        sys.stdout.flush()
        
        tempx = sx * pheno[:, None]
        sx_totals = sx.sum(axis=0)               # (b,)
        casex_totals = tempx.sum(axis=0)         # (b,)
        sy_totals = sy_full.sum(axis=0)               # (s,)
        caseY_totals = (sy_full * pheno[:, None]).sum(axis=0)  # (s,)
        
        # For symmetric models (RR/DD) we only need the strict lower triangle of the full
        # s x s matrix - everything else is filled in by mirroring in run(). Compute the
        # (local_row, global_col) coordinates of that triangle once, for this worker's row band.
        if symmetric_flag:
            tril_rows, tril_cols = np.tril_indices(i2 - i1, i1 - 1, s)
            sel = (tril_rows, tril_cols)
            out_rows = tril_rows + i1
            out_cols = tril_cols
        else:
            b = sx.shape[1]
            rr, cc = np.meshgrid(np.arange(b), np.arange(s), indexing='ij')
            sel = (rr.ravel(), cc.ravel())
            out_rows = sel[0] + i1
            out_cols = sel[1]

        # cache = hc.HygeCache(population_size, case_size, control_size)

        g11 = sx.T @ sy_full      # (b, s) - matmul #1
        x11 = tempx.T @ sy_full   # (b, s) - matmul #2
        
        g10 = sx_totals[:, None] - g11
        g01 = sy_totals[None, :] - g11
        g00 = population_size - sx_totals[:, None] - sy_totals[None, :] + g11
        
        x10 = casex_totals[:, None] - x11
        x01 = caseY_totals[None, :] - x11
        x00 = case_size - casex_totals[:, None] - caseY_totals[None, :] + x11
        
        # xp_* = g_* - x_* because pheno_res = 1 - pheno is linear in the counts above.
        xp11 = g11 - x11
        xp10 = g10 - x10
        xp01 = g01 - x01
        xp00 = g00 - x00
        
        g11_v, x11_v = g11[sel], x11[sel]
        g10_v, x10_v = g10[sel], x10[sel]
        g01_v, x01_v = g01[sel], x01[sel]
        g00_v, x00_v = g00[sel], x00[sel]
        xp11_v, xp10_v = xp11[sel], xp10[sel]
        xp01_v, xp00_v = xp01[sel], xp00[sel]
        
        risk_score = helper_score_from_counts(cache, g11_v, x11_v, g10_v, x10_v,
                                        g01_v, x01_v, g00_v, x00_v, alpha1, alpha2, True, pool, n_workers)
        prot_score = helper_score_from_counts(cache, g11_v, xp11_v, g10_v, xp10_v,
                                        g01_v, xp01_v, g00_v, xp00_v, alpha1, alpha2, False, pool, n_workers)

        nz_r = risk_score != 0
        nz_p = prot_score != 0
        
        risk_rows = out_rows[nz_r]
        risk_cols = out_cols[nz_r]
        risk_vals = risk_score[nz_r]
        prot_rows = out_rows[nz_p]
        prot_cols = out_cols[nz_p]
        prot_vals = prot_score[nz_p]
        
        results.append((risk_rows, risk_cols, risk_vals, prot_rows, prot_cols, prot_vals))
        print(f"\t DONE - {datetime.now() - t1}")

    risk_rows = np.concatenate([ r[0] for r in results ])
    risk_cols = np.concatenate([ r[1] for r in results ])
    risk_vals = np.concatenate([ r[2] for r in results ])
    prot_rows = np.concatenate([ r[3] for r in results ])
    prot_cols = np.concatenate([ r[4] for r in results ])
    prot_vals = np.concatenate([ r[5] for r in results ])

    result_risk = scipy.sparse.coo_array((risk_vals, (risk_rows, risk_cols)), shape=(s, s)).tocsr()
    result_protective = scipy.sparse.coo_array((prot_vals, (prot_rows, prot_cols)), shape=(s, s)).tocsr()

    if model == 'RR' or model == 'DD':
        # only the strict lower triangle was computed - mirror it. Upper triangle of
        # result_risk is all zero by construction, so addition doesn't double-count.
        result_risk = result_risk + result_risk.T
        result_protective = result_protective + result_protective.T
    else:
        # RD: full matrix was computed (no triangle dedup) but isn't symmetric by
        # construction - take elementwise max with transpose, zero the diagonal.
        result_risk = result_risk.maximum(result_risk.T)
        result_risk.setdiag(0)
        result_risk.eliminate_zeros()
        
        result_protective = result_protective.maximum(result_protective.T)
        result_protective.setdiag(0)
        result_protective.eliminate_zeros()

    network = InteractionNetwork.InteractionNetwork(result_risk, result_protective, None, None)

    with open(output_name, 'wb') as final:
        pickle.dump(network, final)   

def numpy_arr_sparseness(arr):
    return np.count_nonzero(arr == 0) / arr.size
    
def combine_max(rr, rd, dd):
    # elementwise max across the three — already sparse-native (unchanged from your code)
    network_max_temp = rr.maximum(dd)
    network_max = network_max_temp.maximum(rd)

    # strict inequalities between sparse arrays stay sparse (unlike >=, <=, ==, !=)
    cond3 = network_max_temp < rd   # rd is strictly the overall max
    cond1 = rr > dd                 # rr > dd

    # "cond1 AND NOT cond3": subtract the overlap instead of negating (negation would densify)
    cond1_final = cond1 - cond1.multiply(cond3)

    # nonzero pattern of the overall max, replacing the dense zero-out step
    pattern = network_max.astype(bool)

    region_13 = cond1_final + cond3
    region_2 = pattern - region_13          # "else" case: dd wins (or tie), within the nonzero pattern

    network_max_id = (cond1_final + region_2.multiply(2) + cond3.multiply(3))
    network_max_id.eliminate_zeros()
    
    return network_max, network_max_id
    
def combine(project_dir, alpha1, alpha2, n_jobs, n_workers, pool, R):
    """Run the three models (RR, RD, DD) and combine their results into a single InteractionNetwork.

    Args:
        project_dir (str): _description_
        alpha1 (float): _description_
        alpha2 (float): _description_
        n_jobs (int): _description_
        n_workers (int): _description_
        pool (multiprocessing.Pool): The process pool to use for parallel processing.
        R (int): _description_
    """
    
    # run(project_dir, 'RR', alpha1, alpha2, n_jobs, n_workers, pool, R)
    # run(project_dir, 'RD', alpha1, alpha2, n_jobs, n_workers, pool, R)
    # run(project_dir, 'DD', alpha1, alpha2, n_jobs, n_workers, pool, R)

    ## load results for 3 models
    with open(f"{project_dir}/intermediate/ssM_mhygessi_RR_R{R}.pkl", 'rb') as rr_file:
        rr_network = pickle.load(rr_file)
    with open(f"{project_dir}/intermediate/ssM_mhygessi_RD_R{R}.pkl", 'rb') as rd_file:
        rd_network = pickle.load(rd_file)
    with open(f"{project_dir}/intermediate/ssM_mhygessi_DD_R{R}.pkl", 'rb') as dd_file:
        dd_network = pickle.load(dd_file)

    risk_max, risk_max_id = combine_max(rr_network.risk, rd_network.risk, dd_network.risk)
    protective_max, protective_max_id = combine_max(rr_network.protective, rd_network.protective, dd_network.protective)

    network = InteractionNetwork.InteractionNetwork(risk_max, protective_max, risk_max_id, protective_max_id)
    
    output_name = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{R}.pkl"
    with open(output_name, 'wb') as final:
        pickle.dump(network, final)


In [ ]:
model = 'RR'
n_jobs = 1
n_workers = 8
R = 0
pool = mp.Pool(processes=n_workers)
run(project_dir, model, alpha1, alpha2, n_jobs, n_workers, pool, R)
pool.close()
pool.join()
# njobs = 16  nworkers = 30 - 1m 21s
# njobs = 10  nworkers = 30 - 1m 22s
# njobs = 8   nworkers = 30 - 1m 18s
# njobs = 6   nworkers = 30 - 1m 18s
# njobs = 4   nworkers = 30 - 1m 20s
# njobs = 2   nworkers = 30 - 1m 22s
# njobs = 1   nworkers = 30 - 1m 16s
# njobs = 1   nworkers = 15 - 1m 28s
# njobs = 1   nworkers = 8  - 2m 5s

# larger n_jobs does not change how long it takes to compute the interaction network, just how much memory is used
# larger n_workers can reduce the time to compute the interaction network, but only up to a certain point

In [ ]:
model = 'combined'
n_jobs = 5
n_workers = 30
R = 0
# pool = mp.Pool(processes=n_workers)
pool = None
combine(project_dir, alpha1, alpha2, n_jobs, n_workers, pool, R)
# pool.close()
# pool.join()



In [ ]:
for i in range(6):
    combine(project_dir, alpha1, alpha2, n_jobs, n_workers, pool, i)

## ComputeStats

In [ ]:
job = 'ComputeStats'



if job == 'ComputeStats':
    if not (model == 'RR' or model == 'RD' or model == 'DD' or model == 'combined' or ssmfile != None):
        sys.exit('wrong model')
        
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"{bpmfile} not found")
        
    snpDataAD = f"{project_dir}/intermediate/SNPdataAD.pkl"
    if not path.exists(snpDataAD):
        sys.exit(snpDataAD + ' not found')
        
    snpDataAR = f"{project_dir}/intermediate/SNPdataAR.pkl"
    if not path.exists(snpDataAR):
        sys.exit(snpDataAR + ' not found')
        
    if ssmfile == None:
        if r < 0:
            if model == 'combined':
                ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{str(i)}.pkl"
            else:
                ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{str(i)}.pkl"
            # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)
            
        else:
            for i in range(r+1):
                if model == 'combined':
                    ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R{str(i)}.pkl"
                else:
                    ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{str(i)}.pkl"
                # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)
                
    else:
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
        # gs.genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_workers, densitycutoff)

### Opus5 high

In [ ]:
import math
import pickle
import multiprocessing as mp
from datetime import datetime

import numpy as np
from scipy.sparse import csr_array, issparse
from scipy.stats import chi2, norm, rankdata

from classes import GenstatsOut, Stats
np.seterr(divide='ignore', invalid='ignore')

# genstats() computes BPM/WPM/PATH statistics. Can be run parallel.
#
# REFACTOR NOTES (mirrors the approach taken in matrix_operations_par.py):
#   - The interaction network is kept as a scipy.sparse.csr_array end to end. Nothing in this
#     module ever materializes a dense (s x s) array, and the old dense sharedctypes.RawArray
#     (plus the np.copy of it made inside every worker) is gone. This is the dominant RAM win:
#     the previous code held one dense s x s copy in the parent plus one per worker.
#   - All of the "sum a submatrix block" loops (bpmgi, wpmgi, bpmsum, wpmsum, and the same
#     sums repeated inside every permutation) are replaced by the indicator-matrix identity
#         sum(mm[ind1, :][:, ind2]) == u1.T @ mm @ u2   with u1/u2 0-1 indicator columns
#     evaluated for many BPMs at once as (mm @ U2).multiply(U1).sum(axis=0). One sparse
#     product now does what was a Python loop over bpm_size fancy-indexed slices. cyadd is
#     no longer needed.
#   - Permutations no longer permute the network. Permuting the columns of mm and then summing
#     a block is identical to leaving mm alone and permuting the *rows of the indicator matrix*
#     on the column side, so each permutation costs one sparse product instead of a rebuild of
#     mm plus bpm_size slice-and-sum calls. The permutation SEQUENCE is reproduced exactly -
#     see snp_permutation_parallel().
#   - PATH degree used to call mannwhitneyu(dist_in, dist_out) once per pathway per permutation
#     on length-s dense vectors. Because dist_in/dist_out always partition the same vector
#     (sumMM, or a permutation of it), the midranks and the tie correction can be computed once
#     and reused: the per-pathway statistic is then just a rank sum, i.e. P.T @ ranks. This is
#     exact, not an approximation.
#   - call_chi2 is a closed-form vectorized 2x2 chi-square instead of bpm_size calls into
#     scipy.stats.chi2_contingency.
#   - The old `tr` list was tested with `if i in tr` inside a loop over bpm_size, i.e. O(n^2).
#     It is now the boolean mask `tr_mask`, derived directly from bpm['ind1size']/['ind2size'].
#   - Dead code removed: pre_comp1/pre_comp2/xs1/xs2 (built, shared, read by the workers, then
#     never used - and xs2 was built from pre_comp1 by copy/paste), the unused `arr`, the bare
#     `wpm_local_pv` expression statement, the datetime/psutil timing scaffolding, and the
#     unused `denisty_wpm` typo'd local.
#   - n_jobs / n_workers: n_jobs splits work into that many *sequential* subsets to cap peak
#     RAM, n_workers is the pool width. In the permutation stage n_jobs subsets the BPMs, not
#     the permutations: the permuted index q is produced exactly as the original produced it
#     and then fed to each subset in turn, so peak RAM falls with n_jobs while the permutation
#     sequence is untouched.
#   - Worker data sharing is by fork() copy-on-write: publish_shared() installs the read-only
#     sparse structures on the module before the pool is created. This is the same platform
#     assumption the previous sharedctypes/RawArray code already made.
#
# BEHAVIOUR NOTES (differences from the old file):
#   - Empirical p-values are REPRODUCIBLE: for a given snpPerms they do not depend on n_workers
#     or n_jobs, so a result can be reproduced on any machine. The original did not have this
#     property - it seeded one RNG stream per worker (PERM_SEED + proc*1000, each of length
#     snpPerms/n_workers), so changing the worker count changed the set of permutations drawn
#     and hence the p-values.
#     The canonical stream here is the original's n_workers=1 stream: one legacy MT19937 seeded
#     PERM_SEED, snpPerms draws, COMPOUNDING (the original reassigned mmtmp, so iteration k acts
#     on the composition of every draw so far). So this matches an original run at n_workers=1
#     exactly, and will differ from an original run at any other worker count -- but those runs
#     were not reproducible to begin with.
#   - Tables that are not valid contingency tables return p = 1 from call_chi2 rather than
#     raising (as chi2_contingency did) or reporting spurious significance. This covers a zero
#     marginal and, importantly, any negative count - see call_chi2().
#   - density_wpm for *non-kept* WPMs still carries the value computed from the binarized
#     network even when binary_flag is False. That is what the original did (the non-binary
#     branch reassigned a misspelled local) and downstream code may depend on it, so it is
#     preserved deliberately rather than "fixed".
#   - ind2keep_bpm follows the original exactly, including the non-binary branch's narrowing
#     to (bpm_local >= -log10(0.05)) and the recomputation of bpmind1/bpmind2 from that
#     narrowed mask. Note the thresholds here differ from the serial genstats.py sibling
#     (that one uses -log10(0.05) and `> minPath` at the chi2 stage); this file follows
#     genstats_perm.py: -log10(0.1) and `>= minPath`.
#
# INPUTS:
#   ssmFile: Interaction networks file in the pickle format.
#   bpmfile: files containing SNP ids for BPM/WPMs in pickle format.
#   binary_flag: If True, interaction scores are binarized for computing BPM/WPM/PATH significances
#   snpPerms: Number of snp permutations used for computing empirical p-values
#   minPath: minimum size for a pathway to be considered as WPM and in BPM.
#   n_jobs: number of sequential chunks the BPM passes are split into (lower peak RAM)
#   n_workers: number of parallel cpu cores the program shoud use (higher throughput)
#
# OUTPUTS:
#   genstats_<ssmFile without extension>.pkl - This pickle file contains a GenstasOut class, which itself contains 2 Stats class oject
#       - protective_stats: Statistics for protective network including ranksum scores,empirical p-values, expected density for BPM/WPMs
#       - risk_stats: Statistics for risk network including ranksum scores,empirical p-values, expected density for BPM/WPMs


PERM_SEED = 349898398

class perm_args:
    def __init__(self, skip, count):
        self.skip = skip
        self.count = count

class par_rank_args:
    def __init__(self, id, rows):
        self.id = id
        self.rows = np.asarray(rows, dtype=np.int64)


# ---------------------------------------------------------------------------
# worker data sharing
# ---------------------------------------------------------------------------
# Replaces init_worker()/init_worker_perm(). Called in the *parent* before the pool is
# created; children inherit the objects through fork() copy-on-write, so nothing large is
# pickled per job and nothing is copied per worker.

_SHARED = {}

def publish_shared(**kwargs):
    _SHARED.update(kwargs)

def clear_shared():
    _SHARED.clear()


# ---------------------------------------------------------------------------
# sparse helpers
# ---------------------------------------------------------------------------

def as_sparse(mm):
    """Coerce an interaction network to a float64 csr_array with no stored zeros."""
    if issparse(mm):
        out = csr_array(mm)
    else:
        out = csr_array(np.asarray(mm, dtype=np.float64))
    if out.data.dtype != np.float64:
        out.data = out.data.astype(np.float64)
    out.eliminate_zeros()
    return out

def binarize(mm, threshold):
    """Sparse equivalent of `mm[mm>=threshold] = 1; mm[mm<1] = 0`."""
    out = mm.copy()
    out.data = (out.data >= threshold).astype(np.float64)
    out.eliminate_zeros()
    return out

def sparse_quantile(mm, q):
    """np.quantile(dense_mm, q) computed from the stored values alone.

    Assumes every stored value is > 0, which holds for these -log10 score matrices.
    """
    total = int(mm.shape[0]) * int(mm.shape[1])
    data = np.sort(mm.data)
    n_zero = total - data.size

    def value_at(k):
        return 0.0 if k < n_zero else float(data[k - n_zero])

    pos = q * (total - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    v_lo = value_at(lo)
    return v_lo + (pos - lo) * (value_at(hi) - v_lo)

def indicator_matrix(index_lists, s):
    """(s x len(index_lists)) 0-1 csr_array; column j marks the SNPs in index_lists[j]."""
    n = len(index_lists)
    if n == 0:
        return csr_array((s, 0), dtype=np.float64)
    parts = [np.asarray(x, dtype=np.int64).ravel() for x in index_lists]
    lengths = np.fromiter((p.size for p in parts), dtype=np.int64, count=n)
    if lengths.sum() == 0:
        return csr_array((s, n), dtype=np.float64)
    rows = np.concatenate(parts)
    cols = np.repeat(np.arange(n, dtype=np.int64), lengths)
    data = np.ones(rows.size, dtype=np.float64)
    return csr_array((data, (rows, cols)), shape=(s, n))

def block_sums(mm, u1, u2):
    """Column-wise sum(mm[ind1_j, :][:, ind2_j]) for paired indicator columns u1/u2."""
    if u1.shape[1] == 0:
        return np.zeros(0)
    return np.asarray((mm @ u2).multiply(u1).sum(axis=0)).ravel()

def tiled_block_sums(mm, u1_tiles, u2_tiles, out, row_perm=None):
    """block_sums over pre-tiled indicator columns, optionally permuting the column side.

    row_perm is applied to the rows of the u2 tiles, which is equivalent to permuting the
    columns of mm (see refactor notes) but costs a reindex instead of rebuilding mm.
    """
    off = 0
    for u1t, u2t in zip(u1_tiles, u2_tiles):
        k = u1t.shape[1]
        u2p = u2t if row_perm is None else u2t[row_perm, :]
        out[off:off + k] = block_sums(mm, u1t, u2p)
        off += k
    return out

def tile_indicators(u, n_parts):
    """Split the indicator columns into n_parts subsets of BPMs.

    This is what n_jobs controls in the permutation stage: each sparse product then handles
    1/n_parts of the BPMs, so the (s x subset) intermediate - the peak allocation - shrinks
    proportionally. The permutation itself is untouched; the same q feeds every subset.
    """
    k = u.shape[1]
    if k == 0:
        return []
    tile = max(1, math.ceil(k / max(int(n_parts), 1)))
    return [u[:, lo:lo + tile] for lo in range(0, k, tile)]


# ---------------------------------------------------------------------------
# statistics helpers
# ---------------------------------------------------------------------------

def call_chi2(table):
    """Vectorized 2x2 chi-square, no continuity correction.

    Input format (per row): f11(bpm interactions) - f10(non-bpm interactions) -
    f01(bpm non-interactions) - f00(non-bpm non-interactions), i.e. [[f11,f10],[f01,f00]].
    Matches scipy.stats.chi2_contingency(obs, correction=False) for well-formed tables.

    Rows that are not a valid contingency table return p = 1, i.e. "no evidence of
    enrichment", which is the only defensible answer for a test of whether interactions
    differ from expectation. Three cases:
      - f11 == 0: the original short-circuited to p = 1 here, so this is unchanged.
      - a zero row/column marginal: chi2_contingency raised ValueError; the closed form
        divides by zero and yields inf/nan. Only reachable if a region is fully saturated.
      - ANY NEGATIVE COUNT: chi2_contingency also raised here. The closed form would happily
        return a tiny p-value (a negative cell inflates the |ad - bc| numerator while the
        denominator stays positive), reporting a malformed table as maximally significant.
        That is meaningless, so these are forced to p = 1 and reported. A negative count
        signals an upstream size/convention bug - e.g. wpmnotgi = wpmsize - wpmgi going
        negative if wpmsize counts unordered pairs while wpmgi (a full block sum) counts
        ordered ones. bpmnotgi is clamped upstream; wpmnotgi is not.
    """
    table = np.asarray(table, dtype=np.float64)
    a, b, c, d = table[:, 0], table[:, 1], table[:, 2], table[:, 3]
    n = a + b + c + d
    with np.errstate(divide='ignore', invalid='ignore'):
        stat = n * (a * d - b * c) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
    results = chi2.sf(stat, 1)

    negative = np.any(table < 0, axis=1)
    invalid = ~np.isfinite(stat) | (a == 0) | negative
    results[invalid] = 1.0

    n_neg = int(np.count_nonzero(negative))
    if n_neg:
        print(f"\twarning: {n_neg} chi2 table row(s) contained a negative count and were set "
              f"to p=1; check the size vs interaction-count pair conventions upstream")
    return results

def tie_sum(values):
    """sum(t^3 - t) over tie groups, in float64 to survive very large groups."""
    if values.size == 0:
        return 0.0
    counts = np.unique(values, return_counts=True)[1]
    counts = counts[counts > 1].astype(np.float64)
    return float(np.sum(counts ** 3 - counts))

def mw_greater(rank_sum_in, n_in, n_out, ties):
    """Normal-approximation Mann-Whitney p-value, alternative='greater', continuity corrected.

    Same formula as the original ranksum() helper and as
    scipy.stats.mannwhitneyu(..., use_continuity=True, alternative='greater'), but driven by a
    precomputed midrank sum so the ranking can be shared across pathways/permutations.
    Accepts scalars or arrays.
    """
    n_in = np.asarray(n_in, dtype=np.float64)
    n_out = np.asarray(n_out, dtype=np.float64)
    n = n_in + n_out
    u = np.asarray(rank_sum_in, dtype=np.float64) - n_in * (n_in + 1.0) / 2.0
    with np.errstate(divide='ignore', invalid='ignore'):
        var = n_in * n_out * (n + 1.0 - ties / (n * (n - 1.0))) / 12.0
        z = (u - n_in * n_out / 2.0 - 0.5) / np.sqrt(var)
    p = norm.sf(z)
    p = np.where(np.isfinite(p) & (var > 0), p, 1.0)
    return p if p.ndim else float(p)

def mw_greater_sparse(nz_in, n_in, nz_out, n_out):
    """Mann-Whitney for two groups whose unstored entries are all zeros.

    nz_in/nz_out are the *stored* (nonzero, positive) values; n_in/n_out are the true group
    sizes. Zeros form one big tie group at the bottom of the ranking, so the statistic is
    exact without ever materializing the zeros.
    """
    z_in = float(n_in) - nz_in.size
    z_out = float(n_out) - nz_out.size
    z_tot = z_in + z_out

    pooled = np.concatenate((nz_in, nz_out)) if nz_out.size else nz_in
    if pooled.size:
        ranks = rankdata(pooled) + z_tot
        rank_sum_in = float(ranks[:nz_in.size].sum())
    else:
        rank_sum_in = 0.0
    rank_sum_in += z_in * (z_tot + 1.0) / 2.0

    ties = tie_sum(pooled)
    if z_tot > 1:
        ties += z_tot ** 3 - z_tot
    return mw_greater(rank_sum_in, n_in, n_out, ties)

def ranksum(x, y):
    """Kept for API compatibility: x is in the bpm, y is out of the bpm."""
    pooled = np.concatenate((y, x))
    ranks = rankdata(pooled)
    return mw_greater(float(ranks[y.shape[0]:].sum()), x.shape[0], y.shape[0], tie_sum(pooled))

def split_indices(rows, n_parts):
    """Split into at most n_parts non-empty contiguous pieces."""
    return [part for part in np.array_split(np.asarray(rows), max(int(n_parts), 1)) if part.size]


# ---------------------------------------------------------------------------
# parallel workers
# ---------------------------------------------------------------------------

def bpm_chi2_parallel(job_arg):
    """bpmgi / path1bggi / path2bggi for a slice of BPMs (binarized network)."""
    mm = _SHARED['mm']
    sumMM = _SHARED['sumMM']
    ind1 = _SHARED['ind1']
    ind2 = _SHARED['ind2']
    keep = _SHARED['tr_keep']

    rows = job_arg.rows
    bpmgi = np.zeros(rows.size)
    path1bggi = np.zeros(rows.size)
    path2bggi = np.zeros(rows.size)

    valid = keep[rows]
    if valid.any():
        sel = rows[valid]
        u1 = indicator_matrix([ind1[i] for i in sel], mm.shape[0])
        u2 = indicator_matrix([ind2[i] for i in sel], mm.shape[0])
        gi = block_sums(mm, u1, u2)
        bpmgi[valid] = gi
        path1bggi[valid] = np.asarray(u1.T @ sumMM).ravel() - gi
        path2bggi[valid] = np.asarray(u2.T @ sumMM).ravel() - gi
    return bpmgi, path1bggi, path2bggi

def parallel_ranksum(job_arg):
    """bpmsum + ranksum p-value for a slice of the kept BPMs (non-binary network)."""
    mm = _SHARED['mm']
    bpmind1 = _SHARED['bpmind1']
    bpmind2 = _SHARED['bpmind2']
    s = mm.shape[1]

    rows = job_arg.rows
    bpmsum_tmp = np.zeros(rows.size)
    bpm_local_tmp = np.ones(rows.size)
    mask = np.zeros(s, dtype=bool)

    for k, i in enumerate(rows):
        id1 = np.asarray(bpmind1[i], dtype=np.int64)
        id2 = np.asarray(bpmind2[i], dtype=np.int64)
        if id1.size < 5 or id2.size < 5:
            continue  # bpmsum 0 / p-value 1, as the old `tr` bookkeeping did

        block = mm[id1, :]
        mask[id2] = True
        inside = mask[block.indices]
        mask[id2] = False

        nz_in = block.data[inside]
        nz_out = block.data[~inside]
        n_in = id1.size * id2.size
        n_out = id1.size * (s - id2.size)

        bpmsum_tmp[k] = nz_in.sum()
        bpm_local_tmp[k] = mw_greater_sparse(nz_in, n_in, nz_out, n_out)
    return bpmsum_tmp, bpm_local_tmp

def snp_permutation_parallel(perm_args):
    """Run `share` SNP permutations and return exceedance counts for BPM/WPM/PATH.

    The permutation SEQUENCE reproduces the original bit for bit. Two details make that work:

    1. The RNG is the legacy global MT19937, seeded per worker as PERM_SEED + id * 1000, drawing
       exactly one np.random.permutation(s) per iteration. Switching to np.random.default_rng
       would draw a different sequence even from the same nominal seed.

    2. The original wrote `mmtmp = mmtmp[:, np.random.permutation(...)]`, REASSIGNING mmtmp, so
       the permutations compound: iteration k operates on mm[:, q_k] where q_k = q_{k-1}[p_k]
       and q_0 = arange(s). Each q_k is still marginally uniform (so the sampled distribution
       was never wrong), but the sequence is a random walk on the symmetric group rather than
       k independent draws. Carrying q forward here reproduces it; drawing fresh permutations
       agrees only on the first iteration.

    Given q, the column-permuted block sum is obtained by gathering the rows of the
    column-side indicator matrix with argsort(q), and the permuted column sums are just the
    original column sums reindexed by q.
    """
    mm = _SHARED['mm']
    u1_tiles = _SHARED['u1_tiles']
    u2_tiles = _SHARED['u2_tiles']
    pw = _SHARED['pw']
    ppath = _SHARED['ppath']
    bpmsum_obs = _SHARED['bpmsum_obs']
    wpmsum_obs = _SHARED['wpmsum_obs']
    path_obs = _SHARED['path_obs']
    col_ranks = _SHARED['col_ranks']
    col_ties = _SHARED['col_ties']
    path_lens = _SHARED['path_lens']
    s = mm.shape[0]

    ## ONE canonical stream, seeded the same way regardless of n_workers or n_jobs, so the
    ## permutations - and therefore the empirical p-values - are reproducible on any hardware.
    ## This is exactly the stream the original produced when run with n_workers=1.
    np.random.seed(PERM_SEED)

    count_bpm = np.zeros(bpmsum_obs.size)
    count_wpm = np.zeros(wpmsum_obs.size)
    count_path = np.zeros(path_obs.size)

    bpmsum_tmp = np.empty(bpmsum_obs.size)
    n_out_path = s - path_lens

    q = np.arange(s)  # cumulative permutation, matching the original's reassignment of mmtmp

    ## Advance to this piece's start by drawing and composing the permutations it is not
    ## evaluating. A draw is well under 1% of an evaluated iteration, so this is close to free,
    ## and it means the stream is identical to a worker that ran the whole share end to end.
    for _ in range(perm_args.skip):
        q = q[np.random.permutation(s)]

    for perm in range(perm_args.count):
        q = q[np.random.permutation(s)]
        inv = np.empty(s, dtype=np.int64)      # inverse by scatter: O(s), not O(s log s)
        inv[q] = np.arange(s, dtype=np.int64)

        # BPM: permuting mm's columns == permuting the rows of the column-side indicators
        tiled_block_sums(mm, u1_tiles, u2_tiles, bpmsum_tmp, row_perm=inv)
        count_bpm += bpmsum_tmp > bpmsum_obs

        # WPM: same block on both sides, so the same trick applies
        if wpmsum_obs.size:
            wpmsum_tmp = block_sums(mm, pw, pw[inv, :])
            count_wpm += wpmsum_tmp > wpmsum_obs

        # PATH degree: the permuted column sums are a permutation of the original ones, so
        # the midranks are known up front and the statistic reduces to a rank sum.
        if path_obs.size:
            rank_in = np.asarray(ppath.T @ col_ranks[q]).ravel()
            p = mw_greater(rank_in, path_lens, n_out_path, col_ties)
            count_path += (-1 * np.log10(p)) > path_obs

    return count_bpm, count_wpm, count_path


# ---------------------------------------------------------------------------
# main routine
# ---------------------------------------------------------------------------

def rungenstats(input_network, bpm, wpm, minPath, binary_flag, snpPerms, n_jobs, n_workers):
    ## inputs:
    ## - input_network: scipy.sparse interaction network (csr_array)
    ## - bpm: bpm dataframe
    ## - wpm: wpm dataframe
    ## - minPath: minimum number of snps in a pathway
    ## - binary_flag: flag to make the interaction network binary
    ## - n_jobs: sequential work chunks (RAM), n_workers: pool width (speed)

    n_jobs = max(int(n_jobs), 1)
    n_workers = max(int(n_workers), 1)
    ctx = mp.get_context('fork')  # workers read the sparse structures copy-on-write

    mm_scores = as_sparse(input_network)
    s = mm_scores.shape[0]

    bpm_size = bpm['size'].values.shape[0]
    bpmsize = bpm['size'].values
    ind1 = bpm['ind1'].values
    ind2 = bpm['ind2'].values
    bpmind1size = bpm['ind1size'].values
    bpmind2size = bpm['ind2size'].values

    wpm_size = wpm['size'].values.shape[0]
    wpmsize = wpm['size'].values
    wpmindsize = wpm['indsize'].values
    ind = wpm['ind'].values

    ## ?Binary  -- mm is the binarized network used for the chi2 stage; mm_scores is kept
    ## alongside it instead of being np.copy()'d (both are sparse, so this is cheap).
    # TODO: should this threshold be tunable or changed?
    if binary_flag:
        # if true, then the network was already binarized with present or not present
        mm = mm_scores
    else:
        # else, binarize with a 0.2 cutoff, but since this is -log10(pvalues) then it is equivalent to a pvalue of 0.63
        # since 0.1 pvalue threshold is used with chi2 marginal significance, maybe that should be used here too?
        # -1.0 * log10(0.1) = 1.0, so maybe use this instead?
        mm = binarize(mm_scores, 0.2)

    sumMM = np.asarray(mm.sum(axis=1)).ravel()

    ## pathway indicator matrix: column a marks the SNPs of pathway a. Reused for every WPM
    ## and PATH statistic below, observed and permuted.
    path_lists = [ np.asarray(x, dtype=np.int64).ravel() for x in ind ]
    path_lens = np.fromiter((p.size for p in path_lists), dtype=np.int64, count=wpm_size).astype(np.float64)
    pmat = indicator_matrix(path_lists, s)

    ### BPM binary chi2
    print("\tBPM chi2: ", end="")
    t1 = datetime.now()
    # bpm genetic interaction counts + background interactions, in n_jobs sequential chunks
    # of n_workers parallel slices. `tr` is now a mask instead of an O(n^2) `in` test.
    tr_mask = (bpmind1size < 5) | (bpmind2size < 5)
    publish_shared(mm=mm, sumMM=sumMM, ind1=ind1, ind2=ind2, tr_keep=~tr_mask)

    bpmgi = np.zeros(bpm_size)
    path1bggi = np.zeros(bpm_size)
    path2bggi = np.zeros(bpm_size)

    with ctx.Pool(processes=n_workers) as pool:
        for chunk in split_indices(np.arange(bpm_size), n_jobs):
            job_args = [par_rank_args(i, part) for i, part in enumerate(split_indices(chunk, n_workers))]
            for j_arg, res in zip(job_args, pool.map(bpm_chi2_parallel, job_args)):
                bpmgi[j_arg.rows] = res[0]
                path1bggi[j_arg.rows] = res[1]
                path2bggi[j_arg.rows] = res[2]
    clear_shared()

    # bpm non interaction
    bpmnotgi = bpmsize - bpmgi
    bpmnotgi[bpmnotgi < 0] = 0
    # non-bpm non-interation
    path1bgsize = bpmind1size * s
    path2bgsize = bpmind2size * s

    path1notgi = path1bgsize - path1bggi - bpmsize
    path2notgi = path2bgsize - path2bggi - bpmsize

    # call chi2
    ## build the tables
    table1 = np.stack((bpmgi, path1bggi, bpmnotgi, path1notgi)).transpose()
    table1[tr_mask, :] = 5
    table2 = np.stack((bpmgi, path2bggi, bpmnotgi, path2notgi)).transpose()
    table2[tr_mask, :] = 5
    ## call chi2
    chi2_bpm_1 = np.log10(call_chi2(table1)) * -1.0
    chi2_bpm_2 = np.log10(call_chi2(table2)) * -1.0
    chi2_bpm_1[tr_mask] = 0
    chi2_bpm_2[tr_mask] = 0

    ## consider under-enriched chi2s
    under1 = bpmgi / (bpmgi + bpmnotgi) < path1bggi / (path1bggi + path1notgi)
    under2 = bpmgi / (bpmgi + bpmnotgi) < path2bggi / (path2bggi + path2notgi)
    chi2_bpm_1[under1] = -1 * chi2_bpm_1[under1]
    chi2_bpm_2[under2] = -1 * chi2_bpm_2[under2]

    ## compute densitites
    density_bpm_local_1 = (bpmgi + path1bggi) / (path1notgi + path1bggi + bpmsize)
    density_bpm_local_2 = (bpmgi + path2bggi) / (path2notgi + path2bggi + bpmsize)

    ## choose the denser (or lower chi2 value)
    dense_index = np.zeros(bpm_size)
    dense_index[chi2_bpm_1 < chi2_bpm_2] = 1
    dense_index[chi2_bpm_1 > chi2_bpm_2] = 2
    dense_index[(dense_index == 0) & (density_bpm_local_1 > density_bpm_local_2)] = 1
    dense_index[(dense_index == 0) & (density_bpm_local_1 < density_bpm_local_2)] = 2

    ## finalize bpm local
    chi2_bpm_local = np.zeros(bpm_size)
    chi2_bpm_local[dense_index == 1] = chi2_bpm_1[dense_index == 1]
    chi2_bpm_local[dense_index == 2] = chi2_bpm_2[dense_index == 2]

    ## keeping track of significant bpms
    ind2keep_bpm = (chi2_bpm_local >= (-1.0 * np.log10(0.1))) & (bpmind1size >= minPath) & (bpmind2size >= minPath)

    ## keeping denser pathway in ind1_new
    swap = dense_index == 2
    ind1_new = np.where(swap, ind2, ind1)
    ind2_new = np.where(swap, ind1, ind2)
    ind1size_new = np.where(swap, bpmind2size, bpmind1size)

    ## pairs to keep
    bpmind1 = ind1_new[ind2keep_bpm]
    bpmind2 = ind2_new[ind2keep_bpm]
    print(f"{ind2keep_bpm.sum()} passed - {str(datetime.now() - t1).split('.')[0]}")

    ###WPM Chi2
    print("\tWPM chi2: ", end="")
    t1 = datetime.now()
    ## one sparse product replaces the per-pathway loop; the diagonal of P.T @ mm @ P is
    ## exactly sum(mm[ind[i], :][:, ind[i]]).
    wpmgi = block_sums(mm, pmat, pmat)
    wpmnotgi = wpmsize - wpmgi
    density_wpm = wpmgi / wpmsize

    ## WPM background size and interactions
    pathbggi = np.asarray(pmat.T @ sumMM).ravel() - wpmgi
    pathbgsize = wpmindsize * s
    pathbgnotgi = pathbgsize - pathbggi - wpmsize

    wpm_table = np.stack((wpmgi, pathbggi, wpmnotgi, pathbgnotgi)).transpose()

    ## call chi2
    chi2_wpm = np.log10(call_chi2(wpm_table)) * -1

    ## consider under-enriched chi2s
    under_wpm = wpmgi / (wpmgi + wpmnotgi) < pathbggi / (pathbggi + pathbgnotgi)
    chi2_wpm[under_wpm] = -1 * chi2_wpm[under_wpm]
    ind2keep_wpm = (chi2_wpm >= -1 * np.log10(0.1))
    print(f"{ind2keep_wpm.sum()} passed - {str(datetime.now() - t1).split('.')[0]}")

    ##### mutual binary - non-binary ends here

    if binary_flag:
        ## compute bpm interaction count and density for the remaining
        bpmsum = np.zeros(bpm_size)
        density_bpm = np.zeros(bpm_size)

        u1_keep = indicator_matrix([np.asarray(x, dtype=np.int64) for x in bpmind1], s)
        u2_keep = indicator_matrix([np.asarray(x, dtype=np.int64) for x in bpmind2], s)
        bpmsum_tmp = np.zeros(bpmind1.shape[0])
        tiled_block_sums(mm, tile_indicators(u1_keep, n_jobs), tile_indicators(u2_keep, n_jobs), bpmsum_tmp)

        density_bpm[ind2keep_bpm] = bpmsum_tmp / bpmsize[ind2keep_bpm]
        bpmsum[ind2keep_bpm] = bpmsum_tmp
        bpm_local = chi2_bpm_local  ## output

        ### WPM density
        wpm_local = chi2_wpm
        wpmsum = np.zeros(wpm_size)
        density_wpm = np.zeros(wpm_size)
        pw = pmat[:, ind2keep_wpm]
        wpmsum_tmp = block_sums(mm, pw, pw)
        density_wpm[ind2keep_wpm] = wpmsum_tmp / wpmsize[ind2keep_wpm]
        wpmsum[ind2keep_wpm] = wpmsum_tmp

    else:
        ## restore non-binary mm
        mm = mm_scores
        sumMM = np.asarray(mm.sum(axis=1)).ravel()
        ## ranksum test
        print("\tBPM ranksum: ", end="")
        t1 = datetime.now()
        bpmsum = np.zeros(bpm_size)
        density_bpm = np.zeros(bpm_size)
        n_keep = bpmind1.shape[0]
        bpmsum_tmp = np.zeros(n_keep)
        bpm_local_tmp = np.ones(n_keep)

        # parallel run for computing ranksum, in n_jobs sequential chunks
        publish_shared(mm=mm, bpmind1=bpmind1, bpmind2=bpmind2)
        with ctx.Pool(processes=n_workers) as pool:
            for chunk in split_indices(np.arange(n_keep), n_jobs):
                job_args = [par_rank_args(i, part) for i, part in enumerate(split_indices(chunk, n_workers))]
                for j_arg, res in zip(job_args, pool.map(parallel_ranksum, job_args)):
                    bpmsum_tmp[j_arg.rows] = res[0]
                    bpm_local_tmp[j_arg.rows] = res[1]
        clear_shared()

        density_bpm[ind2keep_bpm] = bpmsum_tmp / bpmsize[ind2keep_bpm]
        bpm_local = np.zeros(bpm_size)
        bpm_local[ind2keep_bpm] = -1 * np.log10(bpm_local_tmp)
        bpmsum[ind2keep_bpm] = bpmsum_tmp
        ## update ind2keep_bpm
        ind2keep_bpm = (bpm_local >= -1 * np.log10(0.05))
        print(f"{ind2keep_bpm.sum()} passed - {str(datetime.now() - t1).split('.')[0]}")

        ### wpm ranksum
        print("\tWPM ranksum: ", end="")
        t1 = datetime.now()
        ## NOTE: density_wpm deliberately keeps its binarized values outside ind2keep_wpm,
        ## matching the original (which assigned to a misspelled `denisty_wpm` here).
        denisty_wpm = np.zeros(wpm_size)  # TODO: original had this typo
        # density_wpm = np.zeros(wpm_size)  # this is the correct one
        wpmsum = np.zeros(wpm_size)
        wpm_local_tmp = np.ones(wpm_size)
        kept_wpm = np.flatnonzero(ind2keep_wpm)
        mask = np.zeros(s, dtype=bool)
        for a in kept_wpm:
            id1 = path_lists[a]
            block = mm[id1, :]
            mask[id1] = True
            inside = mask[block.indices]
            mask[id1] = False
            nz_in = block.data[inside]
            nz_out = block.data[~inside]
            wpmsum[a] = nz_in.sum()
            wpm_local_tmp[a] = mw_greater_sparse(nz_in, id1.size * id1.size,
                                                 nz_out, id1.size * (s - id1.size))
        density_wpm[ind2keep_wpm] = wpmsum[ind2keep_wpm] / wpmsize[ind2keep_wpm]
        wpm_local = np.zeros(wpm_size)
        wpm_local[ind2keep_wpm] = -1 * np.log10(wpm_local_tmp[ind2keep_wpm])
        ind2keep_wpm = (wpm_local >= -1 * np.log10(0.05))
        print(f"{ind2keep_wpm.sum()} passed - {str(datetime.now() - t1).split('.')[0]}")

    print("\tComputing expected densities ", end="")
    t1 = datetime.now()
    ## Recomputed here, at the same point the original does it, so that the non-binary branch's
    ## narrowed ind2keep_bpm is reflected in the pairs used from here on (the permutation stage
    ## below in particular). Nothing between the branch and this line uses them.
    bpmind1 = ind1_new[ind2keep_bpm]
    bpmind2 = ind2_new[ind2keep_bpm]
    ## compute expected bpm density -- vectorized per chunk instead of one gather per BPM
    density_bpm_expected = np.zeros(bpm_size)
    for chunk in split_indices(np.arange(bpm_size), n_jobs):
        u = indicator_matrix([np.asarray(ind1_new[i], dtype=np.int64) for i in chunk], s)
        lens = ind1size_new[chunk].astype(np.float64)
        totals = np.asarray(u.T @ sumMM).ravel()
        with np.errstate(divide='ignore', invalid='ignore'):
            vals = totals / (s * lens)
        density_bpm_expected[chunk] = np.where(lens > 0, vals, 0.0)

    ## compute expected wpm density
    with np.errstate(divide='ignore', invalid='ignore'):
        density_wpm_expected = np.asarray(pmat.T @ sumMM).ravel() / (s * path_lens)
    density_wpm_expected[path_lens == 0] = 0.0

    ## path degree -- dist_in/dist_out always partition sumMM, so rank once and reuse
    row_ranks = rankdata(sumMM)
    row_ties = tie_sum(sumMM)
    rank_in = np.asarray(pmat.T @ row_ranks).ravel()
    path_degree = -1 * np.log10(mw_greater(rank_in, path_lens, s - path_lens, row_ties))
    ind2keep_path = (path_degree >= -1 * np.log10(0.1))
    print(f"- {str(datetime.now() - t1).split('.')[0]}")

    ## random snp permutation to compute emirical p-value for the significant bpms
    print("\tSNP permutation ", end="")
    t1 = datetime.now()
    np.random.seed(PERM_SEED)  # inert (every worker reseeds), but the original set it here
    bpm_local_pv = np.ones(bpm_size)
    wpm_local_pv = np.ones(wpm_size)
    path_degree_pv = np.ones(wpm_size)

    ## bpmind1/bpmind2 already reflect the final ind2keep_bpm (recomputed above, as in the
    ## original), so the kept pairs, bpmsum baseline and count_bpm all have the same length.
    u1_keep = indicator_matrix([np.asarray(x, dtype=np.int64) for x in bpmind1], s)
    u2_keep = indicator_matrix([np.asarray(x, dtype=np.int64) for x in bpmind2], s)

    ## permuted block sums are compared against the observed ones on the same network
    bpmsum_obs = np.zeros(bpmind1.shape[0])
    tiled_block_sums(mm, tile_indicators(u1_keep, n_jobs), tile_indicators(u2_keep, n_jobs), bpmsum_obs)
    pw = pmat[:, ind2keep_wpm]
    wpmsum_obs = block_sums(mm, pw, pw)

    ## the permutation compares against permuted *column* sums, so rank those
    col_sums = np.asarray(mm.sum(axis=0)).ravel()

    publish_shared(
        mm=mm,
        u1_tiles=tile_indicators(u1_keep, n_jobs),
        u2_tiles=tile_indicators(u2_keep, n_jobs),
        pw=pw,
        ppath=pmat[:, ind2keep_path],
        bpmsum_obs=bpmsum_obs,
        wpmsum_obs=wpmsum_obs,
        path_obs=path_degree[ind2keep_path],
        col_ranks=rankdata(col_sums),
        col_ties=tie_sum(col_sums),
        path_lens=path_lens[ind2keep_path],
    )

    count_bpm = np.zeros(bpmind1.shape[0])
    count_wpm = np.zeros(int(np.sum(ind2keep_wpm)))
    count_path = np.zeros(int(np.sum(ind2keep_path)))

    ## assign parallel job args -- the original split, preserved exactly: proc 0 takes the
    ## remainder, every other proc takes floor(snpPerms/n_workers). n_jobs is NOT applied here;
    ## any other split changes each worker's share and therefore its seed.
    ## snpPerms permutations are drawn from a single stream, so the work is split by simply
    ## cutting that stream into contiguous pieces. A piece advances to its start by drawing (not
    ## evaluating) the permutations it skips, which is why the split can be chosen freely: every
    ## permutation still comes from the same stream at the same position no matter how many
    ## pieces there are. n_workers and n_jobs therefore change only the speed, never the result.
    ##
    ## Pieces are equal in evaluated count, which is also the minimum total fast-forward. The
    ## last piece skips the whole stream, but a draw is a fraction of a percent of an evaluated
    ## iteration, and that skip runs while the other workers are doing real work.
    pieces = [pc for pc in np.array_split(np.arange(snpPerms), n_workers) if pc.size]
    job_args = [perm_args(int(pc[0]), int(pc.size)) for pc in pieces]

    with ctx.Pool(processes=n_workers) as pool:
        results = pool.map(snp_permutation_parallel, job_args)
    # combine results
    for res in results:
        count_bpm = count_bpm + res[0]
        count_wpm = count_wpm + res[1]
        count_path = count_path + res[2]
    clear_shared()
    print(f"- {str(datetime.now() - t1).split('.')[0]}")

    bpm_local_pv[ind2keep_bpm] = (count_bpm + 1) / snpPerms
    wpm_local_pv[ind2keep_wpm] = (count_wpm + 1) / snpPerms
    path_degree_pv[ind2keep_path] = (count_path + 1) / snpPerms

    return bpm_local, bpm_local_pv, density_bpm, density_bpm_expected, dense_index, wpm_local, wpm_local_pv, density_wpm, density_wpm_expected, path_degree, path_degree_pv

def genstats(in_ssmfile, out_ssmfile, bpmfile, binary_flag, snpPerms, minPath, n_jobs, n_workers, netDensity):
    ### load bpmfile
    with open(bpmfile, 'rb') as pklin:
        bpm_obj = pickle.load(pklin)
    bpm = bpm_obj.bpm
    wpm = bpm_obj.wpm
    print(f"\tloaded {bpm.shape[0]:,} BPMs and {wpm.shape[0]} WPMs")

    ### load interaction network
    with open(in_ssmfile, 'rb') as pklin:
        network = pickle.load(pklin)
    p_network = as_sparse(network.protective)
    r_network = as_sparse(network.risk)
    # TODO: remove this when only working with sparse arrays
    del network
    
    print(f"\t{p_network.shape[0] * p_network.shape[1]:,} entries in the SNP-SNPinteraction network")
    p_density = p_network.nnz / (p_network.shape[0] * p_network.shape[1]) * 100
    print(f"\t{p_density:.2f}% of the entries are nonzero in protective network")
    r_density = r_network.nnz / (r_network.shape[0] * r_network.shape[1]) * 100
    print(f"\t{r_density:.2f}% of the entries are nonzero in risk network")

    if binary_flag:
        if netDensity is None:
            ## every stored value is > 0, so this is just "set the stored values to 1"
            p_network = binarize(p_network, 0)
            r_network = binarize(r_network, 0)
        else:
            p_cutoff = sparse_quantile(p_network, 1 - netDensity)
            r_cutoff = sparse_quantile(r_network, 1 - netDensity)
            ## a cutoff of 0 would binarize the zeros too, i.e. densify to an all-ones s x s
            ## matrix. That is unrepresentable sparsely (and almost certainly not intended),
            ## so fall back to keeping the stored entries and say so.
            for name, cutoff in (('protective', p_cutoff), ('risk', r_cutoff)):
                if cutoff <= 0:
                    print(f"\twarning: netDensity={netDensity} puts the {name} cutoff at "
                          f"{cutoff}; keeping all nonzero entries instead of densifying")
            p_network = binarize(p_network, max(p_cutoff, np.finfo(np.float64).tiny))
            r_network = binarize(r_network, max(r_cutoff, np.finfo(np.float64).tiny))
            
    print(f"running genstats on protective network")
    p_results = rungenstats(p_network, bpm, wpm, minPath, binary_flag, snpPerms, n_jobs, n_workers)
    p_stats = Stats.Stats(*p_results)
    
    print(f"running genstats on risk network")
    r_results = rungenstats(r_network, bpm, wpm, minPath, binary_flag, snpPerms, n_jobs, n_workers)
    r_stats = Stats.Stats(*r_results)
    print()
    out_obj = GenstatsOut.GenstatsOut(p_stats, r_stats)
    tmp = out_ssmfile.split('/')
    tmp[-1] = 'genstats_' + tmp[-1]
    outputfile = '/'.join(tmp)
    with open(outputfile, 'wb') as final:
        pickle.dump(out_obj, final)


In [ ]:
input_project_dir = "/home/fisch872/mat/projects/BridGE-Python/testdata"
output_project_dir = "/home/fisch872/mat/projects/BridGE-Python-AoU/testing"
model = "combined"

bpmfile = f"{input_project_dir}/intermediate/BPMind.pkl"
binaryNetwork = False
snpPerms = 100
minPath = 10
n_jobs = 10
n_workers = 30
densitycutoff = None

R = 5
for i in range(R+1):
    print(f'Computing statistics on {model}_R{i}')
    input_ssmfile = f"{input_project_dir}/intermediate/ssM_mhygessi_{model}_R{i}.pkl"
    output_ssmfile = f"{output_project_dir}/intermediate/ssM_mhygessi_{model}_R{i}.pkl"
    genstats(input_ssmfile, output_ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_jobs, n_workers, densitycutoff)

# previous version ran in ~18hr
# refactored version ran in ~2hr
# permutations won't be exactly the same as the original

In [ ]:
input_project_dir = "/home/fisch872/mat/projects/BridGE-Python/testdata"
model = "combined"
R = 0

ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R{R}.pkl"
bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
binaryNetwork = False
snpPerms = 100
minPath = 10
n_jobs = 2
n_workers = 30
densitycutoff = None

print(f'Computing statistics: R={R} model={model}')
genstats(ssmfile, bpmfile, binaryNetwork, snpPerms, minPath, n_jobs, n_workers, densitycutoff)
# njobs: 10 | n_workers: 30 | 26m | opus-5 high refactor | snp perms 18m
# njobs: 2  | n_workers: 30 | 21m | opus-5 high refactor | snp perms 15m
# njobs: 10 | n_workers: 10 | 30m | opus-5 high refactor | snp perms 20m
# njobs: 2  | n_workers: 10 | 29m | opus-5 high refactor | snp perms 19m

Differences noted between n_jobs and n_workers

- More n_jobs makes SNP perm run longer, but no real difference on all other calculations (only change would be in BPM calculations).
- However, the above changes when less n_workers and less n_jobs increases time to compute SNP perms
- More n_workers decrease time to compute BPM and SNP perm stats, other calcs not affected much at all, but only if n_jobs is small. With larger n_jobs (to reduce RAM usage) more workers won't as much
- Best overall time was jobs 2 and workers 30, which I think overall makes sense and already drastically reduces RAM usage but a large factor compared to jobs=1. Therefore, I will hardcode setting n_jobs to 2 if any value less than 2 is given

n_jobs = 10

n_workers = 30

- running genstats on protective network
    - BPM chi2 - 0:01:15
    - WPM chi2 - 0:00:02
    - BPM ranksum - 0:01:29
    - WPM ranksum - 0:00:05
    - Computing expected densities - 0:00:10
    - SNP permutation - 0:11:31

- running genstats on risk network
    - BPM chi2 - 0:01:14
    - WPM chi2 - 0:00:02
    - BPM ranksum - 0:01:05
    - WPM ranksum - 0:00:04
    - Computing expected densities - 0:00:10
    - SNP permutation - 0:07:43

n_jobs = 10

n_workers = 10

- running genstats on protective network
    - BPM chi2 - 0:01:57
    - WPM chi2 - 0:00:04
    - BPM ranksum - 0:02:38
    - WPM ranksum - 0:00:05
    - Computing expected densities - 0:00:10
    - SNP permutation - 0:11:37

- running genstats on risk network
    - BPM chi2 - 0:02:00
    - WPM chi2 - 0:00:04
    - BPM ranksum - 0:01:49
    - WPM ranksum - 0:00:04
    - Computing expected densities - 0:00:11
    - SNP permutation - 0:07:46

n_jobs = 2
n_workers = 30

regular block tile

- running genstats on protective network
    - BPM chi2 - 0:01:29
    - WPM chi2 - 0:00:02
    - BPM ranksum - 0:01:26
    - WPM ranksum - 0:00:05
     -Computing expected densities - 0:00:09
     -SNP permutation - 0:08:51

- running genstats on risk network
    - BPM chi2 - 0:01:34
    - WPM chi2 - 0:00:03
    - BPM ranksum - 0:00:58
    - WPM ranksum - 0:00:04
    - Computing expected densities - 0:00:10
    - SNP permutation - 0:05:47


doubled block tile

running genstats on protective network
	BPM chi2: 98526 passed - 0:01:30
	WPM chi2: 375 passed - 0:00:04
	BPM ranksum: 94820 passed - 0:01:26
	WPM ranksum: 307 passed - 0:00:05
	Computing expected densities - 0:00:09
	SNP permutation - 0:09:13

running genstats on risk network
	BPM chi2: 65928 passed - 0:01:32
	WPM chi2: 312 passed - 0:00:02
	BPM ranksum: 63303 passed - 0:00:55
	WPM ranksum: 253 passed - 0:00:04
	Computing expected densities - 0:00:10
	SNP permutation - 0:06:15

n_jobs = 2

n_workers = 10

- running genstats on protective network
    - BPM chi2 - 0:02:42
    - WPM chi2 - 0:00:04
    - BPM ranksum - 0:02:40
    - WPM ranksum - 0:00:05
    - Computing expected densities - 0:00:09
    - SNP permutation - 0:11:17

- running genstats on risk network
    - BPM chi2 - 0:02:40
    - WPM chi2 - 0:00:04
    - BPM ranksum - 0:01:38
    - WPM ranksum - 0:00:04
    - Computing expected densities - 0:00:10
    - SNP permutation - 0:07:52

## ComputeFDR

In [ ]:
job = 'ComputeFDR'



if job == 'ComputeFDR':
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"{bpmfile} not found")
        
    if ssmfile == None:
        if model == 'combined':
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
        else:
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R0.pkl"
    else:
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
    if not path.exists(ssmfile):
        sys.exit(f"{ssmfile} not found")
        
    # fdr.fdrsampleperm(ssmfile, bpmfile, pval_cutoff, minPath, sample_perms)

In [ ]:
import pickle

import numpy as np
import pandas as pd

from classes import fdrresultsclass as fdrr
from classes import GenstatsOut
from classes import Stats


# fdrsampleperm() computes False Discovery Rates for BPM/WPM/PATH modules
#
# INPUTS:
#   ssmFile: Interaction networks file(path to file) in the pickle format.
#   BPMindFile: file containing SNP ids for BPM/WPMs in pickle format.
#   pcut: p-value cutoff for BPM/WPM/PATH to be considered significant and to be in FDR computing process
#   minPath: minimum size for a pathway to be considered as WPM and in BPM.
#   N: Number of random networks
#
# OUTPUTS:
#   results_<ssmFile without extension>.pkl - This pickle file contains a fdrresultclass class with fields:
#       - bpm_pv: empirical p-values for BPMs
#       - wpm_pv: empirical p-values for WPMs
#       - path_pv: empirical p-values for PATHs
#       - bpm_ranksum: -log10 ranksum p-values for BPMs
#       - wpm_ranksum: -log10 ranksum p-values for WPMs
#       - path_ranksum: -log10 ranksum p-values for PATHs
#       - fdrbpm2: FDR for BPMs
#       - fdrwpm2: FDR for WPMs
#       - fdrpath2: FDR for PATHs
#
#
# REFACTOR NOTES
#   Statistics are carried as plain (n_rows, N+1) float arrays instead of DataFrames plus lists
#   of column names. Column 0 is the real network, columns 1..N the random ones. Row r is the
#   protective copy of module r and row r + n_modules the risk copy, exactly as the original
#   np.concatenate((protective, risk)) laid them out.
#
#   calculate_fdr() was O(k * n_rows * N) pandas comparisons: for each of the k significant
#   modules it rescanned every module in every network. At 2,707,670 BPM rows, N=20 and
#   k ~ 1.4e5 that is ~7e12 element comparisons through the pandas layer. All four counts
#   (m1/m2/n1/n2) are 2-D dominance counts, so they now come from one sweep over the distinct
#   query p-values -- see _dominance_counts(). The two monotonicity corrections, also written
#   as O(k^2) rescans, are a suffix minimum and a 2-D dominance minimum. Nothing is
#   approximated: all counts are exact, verified against the original loops.
#
#   No multiprocessing. After vectorization the stage is a handful of sorts and cumulative
#   sums; workers would cost more in pickling than they save. The three calculate_fdr() calls
#   are independent and are the place to fan out if profiling ever disagrees.
#
#   Bug fixed: the while-loop collecting significant rows advanced via
#   `vals.loc[valid_row][first_pv_col] = np.nan`, which is chained assignment. Under pandas
#   copy-on-write that writes to a throwaway temporary, the row is never cleared, and the
#   loop spins forever. It is a boolean mask now.
#
#   Bug fixed: the output path was `open('/'.join(outfilename), ...)`, but outfilename is
#   already a string, so join() inserted a slash between every character.


# ---------------------------------------------------------------------------
# helpers for calculate_fdr
# ---------------------------------------------------------------------------

def _frame(values, name):
    return pd.DataFrame({name: np.asarray(values, dtype=np.float64)})


def _stack(data):
    return np.column_stack(list(data.values())).astype(np.float64, copy=False)


def _dominance_counts(pv, s, qpv, qs):
    ## Exact 2-D dominance counts of a point cloud against the k queries:
    ##     m[i] = #{j : pv[j] <= qpv[i]}
    ##     n[i] = #{j : pv[j] <= qpv[i] and s[j] >= qs[i]}
    ##
    ## Sweeps the distinct query p-values in ascending order. Points are bucketed by the
    ## distinct query score thresholds, so once a level's points are folded into the
    ## histogram a single reverse cumulative sum answers every query at that level.
    ## Empirical p-values are multiples of 1/snpPerms, so the number of levels is bounded
    ## by pcut * snpPerms + 1 (~501 at pcut=0.05, snpPerms=10000).

    levels, level_of_query = np.unique(qpv, return_inverse=True)
    thresholds, bin_of_query = np.unique(qs, return_inverse=True)
    n_bins = thresholds.size + 1

    ## a point joins the sweep at the first level >= its p-value (points past the last level
    ## never join), and satisfies s >= thresholds[j] for every j below its own bucket.
    ## entry_level is narrowed because argsort's radix sort makes fewer passes on a smaller
    ## dtype -- worth ~2x on the sort at 5e7 points.
    level_dtype = np.int16 if levels.size < np.iinfo(np.int16).max else np.int32
    entry_level = np.searchsorted(levels, pv, side='left').astype(level_dtype, copy=False)
    bucket = np.searchsorted(thresholds, s, side='right')

    p_order = np.argsort(entry_level, kind='stable')
    p_start = np.searchsorted(entry_level[p_order], np.arange(levels.size + 1), side='left')
    q_order = np.argsort(level_of_query, kind='stable')
    q_start = np.searchsorted(level_of_query[q_order], np.arange(levels.size + 1), side='left')

    hist = np.zeros(n_bins, dtype=np.int64)
    m = np.empty(qpv.size, dtype=np.int64)
    n = np.empty(qpv.size, dtype=np.int64)
    total = 0

    for level in range(levels.size):
        joining = p_order[p_start[level]:p_start[level + 1]]
        if joining.size:
            hist += np.bincount(bucket[joining], minlength=n_bins)
            total += joining.size
        queries = q_order[q_start[level]:q_start[level + 1]]
        if queries.size:
            at_or_above = np.cumsum(hist[::-1])[::-1]
            m[queries] = total
            n[queries] = at_or_above[bin_of_query[queries] + 1]

    return m, n


def _suffix_min(pv, f):
    ## out[i] = min{f[j] : pv[j] >= pv[i]}, ties in pv included.
    ##
    ## Replaces the original's O(k^2) rescan. That rescan wrote results back into the array
    ## it was scanning, but because it walked the values in descending order of f every
    ## update only ever replaced a value with the minimum over a subset of the range being
    ## minimised, which leaves the plain suffix minimum below unchanged.
    order = np.argsort(pv, kind='stable')
    suffix = np.minimum.accumulate(f[order][::-1])[::-1]
    return suffix[np.searchsorted(pv[order], pv, side='left')]


def _dominance_min(pv, s, f):
    ## out[i] = min{f[j] : pv[j] >= pv[i] and s[j] <= s[i]}, ties on both axes included.
    ##
    ## Sweeps the distinct p-values from high to low, so every point with pv >= the current
    ## level is already folded into a per-bucket running minimum over s; the prefix minimum
    ## of that array then answers every query at the level. Same in-place-update argument as
    ## _suffix_min, so this matches the original O(k^2) loop exactly.
    levels, level_of = np.unique(pv, return_inverse=True)
    thresholds, bin_of = np.unique(s, return_inverse=True)

    order = np.argsort(level_of, kind='stable')
    start = np.searchsorted(level_of[order], np.arange(levels.size + 1), side='left')

    best = np.full(thresholds.size, np.inf)
    out = np.empty(f.size, dtype=np.float64)

    for level in range(levels.size - 1, -1, -1):
        group = order[start[level]:start[level + 1]]
        np.minimum.at(best, bin_of[group], f[group])
        out[group] = np.minimum.accumulate(best)[bin_of[group]]

    return out


# ---------------------------------------------------------------------------
# Main funcs for calculate_fdr
# ---------------------------------------------------------------------------

def calculate_fdr(sdf, pvdf, pcut, N, type):
    ## inputs:
    ## - sdf, pvdf: (n_rows, N+1) arrays of ranksum scores and empirical p-values,
    ##   column 0 = real network, columns 1..N = random networks
    ## - pcut: p-value cutoff for a module to enter the FDR computation
    ## - N: number of random networks
    ## - type: 'bpm', 'wpm' or 'path'; names the output columns and selects whether the
    ##   fdr2 correction compares raw or rounded ranksum scores
    ## returns two single-column DataFrames, f'{type}1' and f'{type}2', one row per module,
    ## 1.0 for modules that did not pass pcut

    sdf1 = sdf[:, 0]
    sdf_rest = sdf[:, 1:]
    pv1 = pvdf[:, 0]
    pv_rest = pvdf[:, 1:]

    ## significant modules, in row order -- replaces the first_valid_index() while loop
    vrows = np.flatnonzero(pv1 <= pcut)
    valid_pvs = pv1[vrows]
    vpv1 = sdf1[vrows]

    rfdr1 = np.ones(pv1.size)
    rfdr2 = np.ones(pv1.size)
    if vrows.size == 0:
        return _frame(rfdr1, type + '1'), _frame(rfdr2, type + '2')

    ## m1/m2: modules at least as significant by empirical p-value, real / random networks
    ## n1/n2: modules at least as significant by BOTH empirical p-value and ranksum score
    ## The original's `.ge(0)` filters are no-ops: scores are -log10(p) >= 0 and every
    ## threshold vpv1[i] is itself one of those scores, so `score >= vpv1[i]` implies it.
    ## The 'bpm' and 'wpm'/'path' branches of the original loop were byte-identical, so
    ## there is only one path here.
    m1, n1 = _dominance_counts(pv1, sdf1, valid_pvs, vpv1)
    m2, n2 = _dominance_counts(pv_rest.ravel(), sdf_rest.ravel(), valid_pvs, vpv1)

    with np.errstate(divide='ignore', invalid='ignore'):
        fdr1 = np.nan_to_num(m2 / (N * m1))
        fdr2 = np.nan_to_num(n2 / (N * n1))

    ## correct FDRs so BPM/WPM/PATH with lower p-vals does not have larger FDRs
    rfdr1[vrows] = _suffix_min(valid_pvs, fdr1)

    ## same for fdr2, but a module is also not allowed a larger FDR than any module that is
    ## worse on both axes. WPM/PATH scores are compared rounded to whole numbers here, and
    ## only here -- the counts above use the raw values, as in the original.
    key = vpv1 if type == 'bpm' else np.round(vpv1)
    rfdr2[vrows] = _dominance_min(valid_pvs, key, fdr2)

    return _frame(rfdr1, type + '1'), _frame(rfdr2, type + '2')


def fdrsampleperm(in_ssmFile, out_ssmFile, pcut, N):
    ## one entry per network, keyed exactly like the original DataFrame columns
    bpm_data, bpm_pv_data = {}, {}
    wpm_data, wpm_pv_data = {}, {}
    path_data, path_pv_data = {}, {}

    for i in range(0, N + 1):
        tssmFile = in_ssmFile.replace("_R0", "_R" + str(i))
        tssm_tmp = tssmFile.split('/')
        tssm_tmp[-1] = 'genstats_' + tssm_tmp[-1]
        genstatsfile = '/'.join(tssm_tmp)

        ## load genstats file
        with open(genstatsfile, "rb") as pklin:
            gs = pickle.load(pklin)

        prot, risk = gs.protective_stats, gs.risk_stats

        ## retrieve bpm/wpm/path stats, protective followed by risk
        bpm_data["bpm" + str(i)] = np.concatenate((prot.bpm_local, risk.bpm_local))
        bpm_pv_data["bpm_pv" + str(i)] = np.concatenate((prot.bpm_local_pv, risk.bpm_local_pv))
        wpm_data["wpm" + str(i)] = np.concatenate((prot.wpm_local, risk.wpm_local))
        wpm_pv_data["wpm_pv" + str(i)] = np.concatenate((prot.wpm_local_pv, risk.wpm_local_pv))
        path_data["path" + str(i)] = np.concatenate((prot.path_degree, risk.path_degree))
        path_pv_data["path_pv" + str(i)] = np.concatenate((prot.path_degree_pv, risk.path_degree_pv))

    ## stack into (n_rows, N+1) arrays; column 0 is the real network
    bpm = _stack(bpm_data)
    bpm_pv = _stack(bpm_pv_data)
    wpm = _stack(wpm_data)
    wpm_pv = _stack(wpm_pv_data)
    path = _stack(path_data)
    path_pv = _stack(path_pv_data)

    # calling calculate_fdr() function to compute FDRs
    fdrBPM1, fdrBPM2 = calculate_fdr(bpm, bpm_pv, pcut, N, 'bpm')
    fdrWPM1, fdrWPM2 = calculate_fdr(wpm, wpm_pv, pcut, N, 'wpm')
    fdrPATH1, fdrPATH2 = calculate_fdr(path, path_pv, pcut, N, 'path')

    bpm_ranksum = _frame(bpm[:, 0], 'bpm_ranksum')
    wpm_ranksum = _frame(wpm[:, 0], 'wpm_ranksum')
    path_ranksum = _frame(path[:, 0], 'path_ranksum')
    bpm_pv = _frame(bpm_pv[:, 0], 'bpm_pv')
    wpm_pv = _frame(wpm_pv[:, 0], 'wpm_pv')
    path_pv = _frame(path_pv[:, 0], 'path_pv')

    ssm_tmp = out_ssmFile.split('/')
    ssm_tmp[-1] = 'results_' + ssm_tmp[-1]
    outfilename = '/'.join(ssm_tmp)
    save_obj = fdrr.fdrrclass(
        bpm_pv, wpm_pv, path_pv,
        bpm_ranksum, wpm_ranksum, path_ranksum,
        fdrBPM1, fdrBPM2, fdrWPM1, fdrWPM2, fdrPATH1, fdrPATH2,
        )

    with open(outfilename, 'wb') as fh:
        pickle.dump(save_obj, fh)



In [ ]:
input_project_dir = "/home/fisch872/mat/projects/BridGE-Python/testdata"
output_project_dir = "/home/fisch872/mat/projects/BridGE-Python-AoU/testing"

in_ssmFile = f"{input_project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
out_ssmFile = f"{output_project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
pcut = 0.05
N = 5

fdrsampleperm(in_ssmFile, out_ssmFile, pcut, N)

## Summarize

In [ ]:
job = 'Summarize'



if job == 'Summarize':
    bpmfile = f"{project_dir}/intermediate/BPMind.pkl"
    if not path.exists(bpmfile):
        sys.exit(f"bpm file not found at: {bpmfile}")
        
    snppathwayfile = f"{project_dir}/intermediate/{snppathwayfile}"
    if not path.exists(snppathwayfile):
        sys.exit(f"snp-pathway mapping file not found at: {snppathwayfile}")
        
    snpgenemappingfile = f"{project_dir}/intermediate/snpgenemapping_{int(mappingDistance/1000)}kb.pkl"
    if not path.exists(snpgenemappingfile):
        sys.exit(f"snpgenemappingfile not found at: {snpgenemappingfile}")
    
    if ssmfile == None:
        imported = False
        if model == 'combined':
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
            resultsfile = f"{project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
        else:
            ssmfile = f"{project_dir}/intermediate/ssM_mhygessi_{model}_R0.pkl"
            resultsfile = f"{project_dir}/intermediate/results_ssM_mhygessi_{model}_R0.pkl"
    else:
        resultsfile = f"{project_dir}/intermediate/results_{ssmfile}"
        ssmfile = f"{project_dir}/intermediate/{ssmfile}"
        imported = True
    if not path.exists(ssmfile):
        sys.exit(f"interaction file not found at: {ssmfile}")
    if not path.exists(resultsfile):
        sys.exit(f"results file not found at: {resultsfile}")

    # cl.collectresults(resultsfile, fdrcut, ssmfile, bpmfile, snppathwayfile, snpgenemappingfile, imported, densitycutoff)


In [ ]:
import os
import pickle

import numpy as np
import pandas as pd

from corefuns import check_BPM_WPM_redundancy as cbwr
from corefuns import get_interaction_pair as gpair
from corefuns import pathway_map as pmap

# Imported so pickle can rebuild the objects stored in resultsfile / bpmindfile.
# from classes import BPMind, FDRstats
from classes import fdrresultsclass

FDR_STEP = 0.05


def _stack(series):
    """Duplicate a per-module series: rows 0..n-1 protective, n..2n-1 risk.

    This is the row layout of the FDR / p-value / ranksum frames in the results
    pickle. Returns a one-column DataFrame with a fresh RangeIndex.
    """
    return pd.concat([series, series], ignore_index=True).to_frame()


def _reorder(frame, order):
    """Put a frame in FDR-sorted order and drop back to a RangeIndex."""
    return frame.reindex(index=order).reset_index(drop=True)


def _fdr_levels(fdrcut):
    """Number of FDR_STEP thresholds in fdrcut, which must be a multiple of it."""
    levels = fdrcut / FDR_STEP
    if abs(levels - round(levels)) > 1e-9:
        raise ValueError(f'fdrcut must be a multiple of {FDR_STEP}, got {fdrcut}.')
    return int(round(levels))


def _effect(ind, n_modules, column):
    """Label each significant module protective or risk.

    Protective modules occupy global indices 0..n_modules-1, risk modules
    n_modules..2*n_modules-1. Keeps ind's index so the labels reorder alongside
    every other column.
    """
    labels = np.where(np.asarray(ind.index) < n_modules, 'protective', 'risk')
    return pd.DataFrame({column: labels}, index=ind.index)


def _group_column(groups_per_level, order, label):
    """Redundant-group label for each reported module.

    check_BPM_WPM_redundancy returns one Series per FDR_STEP threshold indexed by
    global module index, so the last one is the level at exactly fdrcut and
    aligns to `order` by label rather than by position.
    """
    labels = groups_per_level[-1].reindex(order)
    missing = int(labels.isna().sum())
    if missing:
        raise ValueError(f'{label}: no redundancy group label for {missing} reported modules.')
    return labels.astype(np.int64).reset_index(drop=True).to_frame('group')


def collectresults(resultsfile, out_resultsfile, fdrcut, ssmfile, bpmindfile, snppathwayfile,
                   snpgenemappingfile, imported_ssm, densitycutoff=None):
    """Collects BPM/WPM/PATH results and exports them to an Excel workbook.

    Also calls out to driver-gene discovery and redundant-module grouping.

    Args:
        resultsfile (str): Pickle file with the FDRs, empirical p-values and
            ranksum scores of the BPM/WPM/PATH modules.
        fdrcut (float): FDR threshold for keeping modules. Must be a multiple
            of 0.05.
        ssmfile (str): Path to the real-network interaction file (pickle).
        bpmindfile (str): Pickle file with the SNP ids for each BPM/WPM.
        snppathwayfile (str): Pickle file mapping SNPs to pathways.
        snpgenemappingfile (str): Pickle file mapping SNPs to genes.
        imported_ssm (bool): True when the interaction network was imported
            rather than computed from genotypes.
        densitycutoff (float, optional): Network density cutoff, passed through
            to get_interaction_pair.

    Returns:
        str: Path to the written workbook, which contains:
            - output_discovery_summary: module counts per FDR threshold.
            - output_noRD_discovery_summary: non-redundant counts per threshold.
            - output_bpm_table / output_wpm_table / output_path_table: the
              modules below fdrcut with their stats and driver genes.
    """
    n_levels = _fdr_levels(fdrcut)
    with open(resultsfile, 'rb') as fh:
        results: FDRstats = pickle.load(fh)
    project_dir = os.path.dirname(os.path.dirname(os.path.abspath(out_resultsfile)))

    fdrBPM, fdrWPM, fdrPATH = results.fdrbpm2, results.fdrwpm2, results.fdrpath2

    ind_bpm = fdrBPM[fdrBPM <= fdrcut].dropna()
    ind_wpm = fdrWPM[fdrWPM <= fdrcut].dropna()
    ind_path = fdrPATH[fdrPATH <= fdrcut].dropna()

    # Narrow the stats frames to the significant modules. Label-based throughout;
    # ind_*.index holds labels from these same frames.
    if not ind_bpm.empty:
        fdrBPM = fdrBPM.loc[ind_bpm.index]
        bpm_pv = results.bpm_pv.loc[ind_bpm.index]
        bpm_ranksum = results.bpm_ranksum.loc[ind_bpm.index]

    if not ind_wpm.empty:
        fdrWPM = fdrWPM.loc[ind_wpm.index]
        wpm_pv = results.wpm_pv.loc[ind_wpm.index]
        wpm_ranksum = results.wpm_ranksum.loc[ind_wpm.index]

    if not ind_path.empty:
        fdrPATH = fdrPATH.loc[ind_path.index]
        path_pv = results.path_pv.loc[ind_path.index]
        path_ranksum = results.path_ranksum.loc[ind_path.index]

    # --- pathway names, sizes, effect direction, driver genes ---------------
    # snppathwayfile is not read here: bridge.py checks it exists and
    # get_interaction_pair loads it (and the geneset it points at) itself.
    if not (ind_bpm.empty and ind_wpm.empty and ind_path.empty):
        with open(bpmindfile, 'rb') as fh:
            bpm_ind: BPMind = pickle.load(fh)
        pathways = bpm_ind.wpm['pathway']
        path_ids = {name: i for i, name in enumerate(pathways)}
        n_bpm = len(bpm_ind.bpm.index)
        n_wpm = len(bpm_ind.wpm.index)

        if not ind_bpm.empty:
            path1 = _stack(bpm_ind.bpm['path1names']).loc[ind_bpm.index]
            path2 = _stack(bpm_ind.bpm['path2names']).loc[ind_bpm.index]
            bpm_size = _stack(bpm_ind.bpm['size']).loc[ind_bpm.index]
            eff_bpm = _effect(ind_bpm, n_bpm, 'eff_bpm')

            bpm_path1_drivers, bpm_path2_drivers, _ = gpair.get_interaction_pair(
                len(ind_bpm), path1, path2, eff_bpm, ssmfile, bpmindfile,
                snppathwayfile, snpgenemappingfile, path_ids, fdrcut,
                imported_ssm, densitycutoff)

        if not ind_wpm.empty:
            path_wpm = _stack(pathways).loc[ind_wpm.index]
            wpm_size = _stack(bpm_ind.wpm['size']).loc[ind_wpm.index]
            eff_wpm = _effect(ind_wpm, n_wpm, 'eff_wpm')

            _, _, wpm_path_drivers = gpair.get_interaction_pair(
                len(ind_wpm), path_wpm, path_wpm, eff_wpm, ssmfile, bpmindfile,
                snppathwayfile, snpgenemappingfile, path_ids, fdrcut,
                imported_ssm, densitycutoff)

        if not ind_path.empty:
            path_path = _stack(pathways).loc[ind_path.index]
            path_size = _stack(bpm_ind.wpm['indsize']).loc[ind_path.index]
            eff_path = _effect(ind_path, n_wpm, 'eff_path')

    # --- redundancy grouping and the pathway map ----------------------------
    (BPM_nosig_noRD, WPM_nosig_noRD, PATH_nosig_noRD,
     BPM_groups, WPM_groups, PATH_groups) = cbwr.check_BPM_WPM_redundancy(
        fdrBPM, fdrWPM, fdrPATH, bpmindfile, fdrcut)

    pmap.draw_map(project_dir, fdrcut, resultsfile, BPM_groups, WPM_groups, PATH_groups)

    # --- output tables ------------------------------------------------------
    # `order` is a stable FDR sort. Group labels are matched to it by module
    # index, so the two orderings do not have to agree.
    # DOUBLE CHECK SORTING WHEN FIXING BPMSIM AND PATHSIM.
    output_bpm_table = output_wpm_table = output_path_table = None

    if not ind_bpm.empty:
        order = fdrBPM.sort_values(kind='stable', by='bpm2').index
        output_bpm_table = pd.concat([
            _reorder(path1, order),
            _reorder(path2, order),
            _group_column(BPM_groups, order, 'BPM'),
            _reorder(fdrBPM, order).round(2),
            _reorder(eff_bpm, order),
            _reorder(bpm_size, order),
            _reorder(bpm_pv, order),
            _reorder(bpm_ranksum, order).round(2),
            _reorder(bpm_path1_drivers, order),
            _reorder(bpm_path2_drivers, order),
        ], axis=1)
        output_bpm_table = output_bpm_table.rename(
            columns={'bpm2': 'fdrBPM', 'size': 'bpm_size'}
        ).sort_values(by=['fdrBPM', 'bpm_pv', 'bpm_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    if not ind_wpm.empty:
        order = fdrWPM.sort_values(kind='stable', by='wpm2').index
        output_wpm_table = pd.concat([
            _reorder(path_wpm, order),
            _group_column(WPM_groups, order, 'WPM'),
            _reorder(fdrWPM, order).round(2),
            _reorder(eff_wpm, order),
            _reorder(wpm_size, order),
            _reorder(wpm_pv, order),
            _reorder(wpm_ranksum, order).round(2),
            _reorder(wpm_path_drivers, order),
        ], axis=1)
        output_wpm_table = output_wpm_table.rename(
            columns={'wpm2': 'fdrWPM', 'size': 'wpm_size'}
        ).sort_values(by=['fdrWPM', 'wpm_pv', 'wpm_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    if not ind_path.empty:
        order = fdrPATH.sort_values(kind='stable', by='path2').index
        output_path_table = pd.concat([
            _reorder(path_path, order),
            _group_column(PATH_groups, order, 'PATH'),
            _reorder(fdrPATH, order).round(2),
            _reorder(eff_path, order),
            _reorder(path_size, order),
            _reorder(path_pv, order),
            _reorder(path_ranksum, order).round(2),
        ], axis=1)
        output_path_table = output_path_table.rename(
            columns={'path2': 'fdrPATH', 'indsize': 'path_size'}
        ).sort_values(by=['fdrPATH', 'path_pv', 'path_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    # --- summary sheets -----------------------------------------------------
    header = ['minfdr'] + [f'fdr{int(round(k * FDR_STEP * 100)):02d}'
                           for k in range(1, n_levels + 1)]

    # One row per module type with results. PATH is checked on its own; the
    # original gated both the WPM and PATH rows on WPM being non-empty.
    rows = [('BPM', fdrBPM, 'bpm2', BPM_nosig_noRD)]
    if not fdrWPM.empty:
        rows.append(('WPM', fdrWPM, 'wpm2', WPM_nosig_noRD))
    if not fdrPATH.empty:
        rows.append(('PATH', fdrPATH, 'path2', PATH_nosig_noRD))

    index = [name for name, _, _, _ in rows]
    discovery = [[frame[col].min()]
                 + [int((frame[col] <= k * FDR_STEP).sum()) for k in range(1, n_levels + 1)]
                 for _, frame, col, _ in rows]
    noRD = [[frame[col].min()] + nosig for _, frame, col, nosig in rows]

    output_discovery_summary = pd.DataFrame(discovery, columns=header, index=index)
    output_noRD_discovery_summary = pd.DataFrame(noRD, columns=header, index=index)

    # --- write --------------------------------------------------------------
    results_dir = os.path.join(project_dir, 'results')
    os.makedirs(results_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(out_resultsfile))[0]
    outfile = os.path.join(results_dir, f'output_{stem}.xlsx')

    sheets = {
        'output_discovery_summary': output_discovery_summary,
        'output_noRD_discovery_summary': output_noRD_discovery_summary,
        'output_bpm_table': output_bpm_table,
        'output_wpm_table': output_wpm_table,
        'output_path_table': output_path_table,
    }
    with pd.ExcelWriter(outfile) as writer:
        for name, table in sheets.items():
            if table is not None:
                table.to_excel(writer, sheet_name=name)


In [ ]:
input_project_dir = "/home/fisch872/mat/projects/BridGE-Python/testdata"
output_project_dir = "/home/fisch872/mat/projects/BridGE-Python-AoU/testing"

in_resultsfile = f"{input_project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
out_resultsfile = f"{output_project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
fdrcut = 0.25
ssmfile = f"{input_project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
bpmindfile = f"{input_project_dir}/intermediate/BPMind.pkl"
snppathwayfile = f"{input_project_dir}/intermediate/snp_pathway_min10_max300.pkl"
snpgenemappingfile = f"{input_project_dir}/intermediate/snpgenemapping_50kb.pkl"
imported_ssm = True
densitycutoff = None

collectresults(
    in_resultsfile,
    out_resultsfile,
    fdrcut,
    ssmfile,
    bpmindfile,
    snppathwayfile,
    snpgenemappingfile,
    imported_ssm,
    densitycutoff,
    )

In [ ]:

import os
import pickle

import numpy as np
import pandas as pd

from corefuns import check_BPM_WPM_redundancy as cbwr
from corefuns import get_interaction_pair as gpair
from corefuns import pathway_map as pmap

# # Imported so pickle can rebuild the objects stored in resultsfile / bpmindfile.
# from classes import bpmindclass  # noqa: F401
# from classes import fdrresultsclass  # noqa: F401

FDR_STEP = 0.05


def _load(path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)


def _stack(series):
    """Duplicate a per-module series: rows 0..n-1 protective, n..2n-1 risk.

    This is the row layout of the FDR / p-value / ranksum frames in the results
    pickle. Returns a one-column DataFrame with a fresh RangeIndex.
    """
    return pd.concat([series, series], ignore_index=True).to_frame()


def _reorder(frame, order):
    """Put a frame in FDR-sorted order and drop back to a RangeIndex."""
    return frame.reindex(index=order).reset_index(drop=True)


def _fdr_levels(fdrcut):
    """Number of FDR_STEP thresholds in fdrcut, which must be a multiple of it."""
    levels = fdrcut / FDR_STEP
    if abs(levels - round(levels)) > 1e-9:
        raise ValueError(f'fdrcut must be a multiple of {FDR_STEP}, got {fdrcut}.')
    return int(round(levels))


def _effect(ind, n_modules, column):
    """Label each significant module protective or risk.

    Protective modules occupy global indices 0..n_modules-1, risk modules
    n_modules..2*n_modules-1. Keeps ind's index so the labels reorder alongside
    every other column.
    """
    labels = np.where(np.asarray(ind.index) < n_modules, 'protective', 'risk')
    return pd.DataFrame({column: labels}, index=ind.index)


def _group_column(groups_per_level, order, label):
    """Redundant-group label for each reported module.

    check_BPM_WPM_redundancy returns one Series per FDR_STEP threshold indexed by
    global module index, so the last one is the level at exactly fdrcut and
    aligns to `order` by label rather than by position.
    """
    labels = groups_per_level[-1].reindex(order)
    missing = int(labels.isna().sum())
    if missing:
        raise ValueError(f'{label}: no redundancy group label for {missing} reported modules.')
    return labels.astype(np.int64).reset_index(drop=True).to_frame('group')


def collectresults(resultsfile, out_resultsfile, fdrcut, ssmfile, bpmindfile, snppathwayfile,
                   snpgenemappingfile, imported_ssm, densitycutoff=None):

    """Collects BPM/WPM/PATH results and exports them to an Excel workbook.

    Also calls out to driver-gene discovery and redundant-module grouping.

    Args:
        resultsfile (str): Pickle file with the FDRs, empirical p-values and
            ranksum scores of the BPM/WPM/PATH modules.
        fdrcut (float): FDR threshold for keeping modules. Must be a multiple
            of 0.05.
        ssmfile (str): Path to the real-network interaction file (pickle).
        bpmindfile (str): Pickle file with the SNP ids for each BPM/WPM.
        snppathwayfile (str): Pickle file mapping SNPs to pathways.
        snpgenemappingfile (str): Pickle file mapping SNPs to genes.
        imported_ssm (bool): True when the interaction network was imported
            rather than computed from genotypes.
        densitycutoff (float, optional): Network density cutoff, passed through
            to get_interaction_pair.

    Returns:
        str: Path to the written workbook, which contains:
            - output_discovery_summary: module counts per FDR threshold.
            - output_noRD_discovery_summary: non-redundant counts per threshold.
            - output_bpm_table / output_wpm_table / output_path_table: the
              modules below fdrcut with their stats and driver genes.
    """
    n_levels = _fdr_levels(fdrcut)

    results = _load(resultsfile)
    project_dir = os.path.dirname(os.path.dirname(os.path.abspath(out_resultsfile)))

    fdrBPM, fdrWPM, fdrPATH = results.fdrbpm2, results.fdrwpm2, results.fdrpath2

    ind_bpm = fdrBPM[fdrBPM <= fdrcut].dropna()
    ind_wpm = fdrWPM[fdrWPM <= fdrcut].dropna()
    ind_path = fdrPATH[fdrPATH <= fdrcut].dropna()

    # Narrow the stats frames to the significant modules. Label-based throughout;
    # ind_*.index holds labels from these same frames.
    if not ind_bpm.empty:
        fdrBPM = fdrBPM.loc[ind_bpm.index]
        bpm_pv = results.bpm_pv.loc[ind_bpm.index]
        bpm_ranksum = results.bpm_ranksum.loc[ind_bpm.index]

    if not ind_wpm.empty:
        fdrWPM = fdrWPM.loc[ind_wpm.index]
        wpm_pv = results.wpm_pv.loc[ind_wpm.index]
        wpm_ranksum = results.wpm_ranksum.loc[ind_wpm.index]

    if not ind_path.empty:
        fdrPATH = fdrPATH.loc[ind_path.index]
        path_pv = results.path_pv.loc[ind_path.index]
        path_ranksum = results.path_ranksum.loc[ind_path.index]

    # --- pathway names, sizes, effect direction, driver genes ---------------
    # snppathwayfile is not read here: bridge.py checks it exists and
    # get_interaction_pair loads it (and the geneset it points at) itself.
    if not (ind_bpm.empty and ind_wpm.empty and ind_path.empty):
        bpm = _load(bpmindfile)
        pathways = bpm.wpm['pathway']
        path_ids = {name: i for i, name in enumerate(pathways)}
        n_bpm = len(bpm.bpm.index)
        n_wpm = len(bpm.wpm.index)

        if not ind_bpm.empty:
            path1 = _stack(bpm.bpm['path1names']).loc[ind_bpm.index]
            path2 = _stack(bpm.bpm['path2names']).loc[ind_bpm.index]
            bpm_size = _stack(bpm.bpm['size']).loc[ind_bpm.index]
            eff_bpm = _effect(ind_bpm, n_bpm, 'eff_bpm')

            bpm_path1_drivers, bpm_path2_drivers, _ = gpair.get_interaction_pair(
                len(ind_bpm), path1, path2, eff_bpm, ssmfile, bpmindfile,
                snppathwayfile, snpgenemappingfile, path_ids, fdrcut,
                imported_ssm, densitycutoff)

        if not ind_wpm.empty:
            path_wpm = _stack(pathways).loc[ind_wpm.index]
            wpm_size = _stack(bpm.wpm['size']).loc[ind_wpm.index]
            eff_wpm = _effect(ind_wpm, n_wpm, 'eff_wpm')

            _, _, wpm_path_drivers = gpair.get_interaction_pair(
                len(ind_wpm), path_wpm, path_wpm, eff_wpm, ssmfile, bpmindfile,
                snppathwayfile, snpgenemappingfile, path_ids, fdrcut,
                imported_ssm, densitycutoff)

        if not ind_path.empty:
            path_path = _stack(pathways).loc[ind_path.index]
            path_size = _stack(bpm.wpm['indsize']).loc[ind_path.index]
            eff_path = _effect(ind_path, n_wpm, 'eff_path')

    # --- redundancy grouping and the pathway map ----------------------------
    (BPM_nosig_noRD, WPM_nosig_noRD, PATH_nosig_noRD,
     BPM_groups, WPM_groups, PATH_groups) = cbwr.check_BPM_WPM_redundancy(
        fdrBPM, fdrWPM, fdrPATH, bpmindfile, fdrcut)

    pmap.draw_map(project_dir, fdrcut, resultsfile, BPM_groups, WPM_groups, PATH_groups)

    # --- output tables ------------------------------------------------------
    # `order` is a stable FDR sort. Group labels are matched to it by module
    # index, so the two orderings do not have to agree.
    # DOUBLE CHECK SORTING WHEN FIXING BPMSIM AND PATHSIM.
    output_bpm_table = output_wpm_table = output_path_table = None

    if not ind_bpm.empty:
        order = fdrBPM.sort_values(kind='stable', by='bpm2').index
        output_bpm_table = pd.concat([
            _reorder(path1, order),
            _reorder(path2, order),
            _group_column(BPM_groups, order, 'BPM'),
            _reorder(fdrBPM, order).round(2),
            _reorder(eff_bpm, order),
            _reorder(bpm_size, order),
            _reorder(bpm_pv, order),
            _reorder(bpm_ranksum, order).round(2),
            _reorder(bpm_path1_drivers, order),
            _reorder(bpm_path2_drivers, order),
        ], axis=1)
        output_bpm_table = output_bpm_table.rename(
            columns={'bpm2': 'fdrBPM', 'size': 'bpm_size'}
        ).sort_values(by=['fdrBPM', 'bpm_pv', 'bpm_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    if not ind_wpm.empty:
        order = fdrWPM.sort_values(kind='stable', by='wpm2').index
        output_wpm_table = pd.concat([
            _reorder(path_wpm, order),
            _group_column(WPM_groups, order, 'WPM'),
            _reorder(fdrWPM, order).round(2),
            _reorder(eff_wpm, order),
            _reorder(wpm_size, order),
            _reorder(wpm_pv, order),
            _reorder(wpm_ranksum, order).round(2),
            _reorder(wpm_path_drivers, order),
        ], axis=1)
        output_wpm_table = output_wpm_table.rename(
            columns={'wpm2': 'fdrWPM', 'size': 'wpm_size'}
        ).sort_values(by=['fdrWPM', 'wpm_pv', 'wpm_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    if not ind_path.empty:
        order = fdrPATH.sort_values(kind='stable', by='path2').index
        output_path_table = pd.concat([
            _reorder(path_path, order),
            _group_column(PATH_groups, order, 'PATH'),
            _reorder(fdrPATH, order).round(2),
            _reorder(eff_path, order),
            _reorder(path_size, order),
            _reorder(path_pv, order),
            _reorder(path_ranksum, order).round(2),
        ], axis=1)
        output_path_table = output_path_table.rename(
            columns={'path2': 'fdrPATH', 'indsize': 'path_size'}
        ).sort_values(by=['fdrPATH', 'path_pv', 'path_ranksum'],
                      ascending=[True, True, False]).reset_index(drop=True)

    # --- summary sheets -----------------------------------------------------
    header = ['minfdr'] + [f'fdr{int(round(k * FDR_STEP * 100)):02d}'
                           for k in range(1, n_levels + 1)]

    # One row per module type with results. PATH is checked on its own; the
    # original gated both the WPM and PATH rows on WPM being non-empty.
    rows = [('BPM', fdrBPM, 'bpm2', BPM_nosig_noRD)]
    if not fdrWPM.empty:
        rows.append(('WPM', fdrWPM, 'wpm2', WPM_nosig_noRD))
    if not fdrPATH.empty:
        rows.append(('PATH', fdrPATH, 'path2', PATH_nosig_noRD))

    index = [name for name, _, _, _ in rows]
    discovery = [[frame[col].min()]
                 + [int((frame[col] <= k * FDR_STEP).sum()) for k in range(1, n_levels + 1)]
                 for _, frame, col, _ in rows]
    noRD = [[frame[col].min()] + nosig for _, frame, col, nosig in rows]

    output_discovery_summary = pd.DataFrame(discovery, columns=header, index=index)
    output_noRD_discovery_summary = pd.DataFrame(noRD, columns=header, index=index)

    # --- write --------------------------------------------------------------
    results_dir = os.path.join(project_dir, 'results')
    os.makedirs(results_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(resultsfile))[0]
    outfile = os.path.join(results_dir, f'output_{stem}.xlsx')

    sheets = {
        'output_discovery_summary': output_discovery_summary,
        'output_noRD_discovery_summary': output_noRD_discovery_summary,
        'output_bpm_table': output_bpm_table,
        'output_wpm_table': output_wpm_table,
        'output_path_table': output_path_table,
    }
    with pd.ExcelWriter(outfile) as writer:
        for name, table in sheets.items():
            if table is not None:
                table.to_excel(writer, sheet_name=name)



In [ ]:
input_project_dir = "/home/fisch872/mat/projects/BridGE-Python/testdata"
output_project_dir = "/home/fisch872/mat/projects/BridGE-Python-AoU/testing"

in_resultsfile = f"{input_project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
out_resultsfile = f"{output_project_dir}/intermediate/results_ssM_mhygessi_combined_R0.pkl"
fdrcut = 0.25
ssmfile = f"{input_project_dir}/intermediate/ssM_mhygessi_combined_R0.pkl"
bpmindfile = f"{input_project_dir}/intermediate/BPMind.pkl"
snppathwayfile = f"{input_project_dir}/intermediate/snp_pathway_min10_max300.pkl"
snpgenemappingfile = f"{input_project_dir}/intermediate/snpgenemapping_50kb.pkl"
imported_ssm = False
densitycutoff = None

collectresults(
    in_resultsfile,
    out_resultsfile,
    fdrcut,
    ssmfile,
    bpmindfile,
    snppathwayfile,
    snpgenemappingfile,
    imported_ssm,
    densitycutoff,
    )